# Tracking High-Speed & Tiny Objects in Sports
### Re-implementing **TrackNetV5** (Tang, Chen & Jiang, arXiv:2512.02789, Dec 2025) and improving it → **TrackNetV5-Lite+**

**Deep Learning Project — Semester VII**

---

**Abstract.** Balls and shuttlecocks in broadcast sports video are *tiny* (often 2–8 px wide), *fast* (motion-blurred into faint streaks), and *frequently occluded* by players. Single-frame object detectors fail on them. The TrackNet family solves this with heatmap regression from a short stack of consecutive frames. Its newest member, **TrackNetV5 (2025)**, adds two modules. *Motion Direction Decoupling (MDD)* turns signed frame differences into learnable attention maps. *Residual-Driven Spatio-Temporal Refinement (R-STR)* is a factorised space–time Transformer head that corrects a draft heatmap.

This notebook does four things:
1. Studies the theory behind the TrackNet line (V1 → V5).
2. Builds a **procedural broadcast-video generator** with ground truth, so no dataset download is needed. It runs on a free Kaggle GPU, or at reduced scale on a laptop CPU. When the real TrackNet tennis dataset is attached, every model is also **fine-tuned on broadcast games 1–8 and tested on unseen games 9–10**.
3. Re-implements **TrackNetV2** (baseline) and a **faithful TrackNetV5**.
4. Proposes **five modifications (M1–M5)**, each motivated by a weakness we identified in the V5 design. Every change is measured in an ablation study covering accuracy, localisation error, parameters, FLOPs, CPU FPS and robustness, with visual analysis throughout.

**Roadmap**
| § | Content |
|---|---|
| 1 | Theory: problem, TrackNet V1→V5, evaluation protocol |
| 2 | Setup & configuration (CPU / GPU profiles) |
| 3 | Synthetic broadcast data generator + dataset analysis |
| 4 | Real TrackNet tennis dataset: game-level split (train 1–8 / test 9–10) |
| 5 | Models: TrackNetV2, TrackNetV5 (faithful), our modifications M1–M4 |
| 6 | Training |
| 7 | Evaluation & ablation study |
| 8 | M5: physics-aware temporal post-processing |
| 9 | Visual analysis + **real-footage fine-tuning & evaluation (§9.5)** |
| 10 | Changes, trade-offs & improvements (summary) |
| 11 | Conclusions, limitations, references |
| 12 | Report assets: tools/environment record + checklist of every figure and table the report needs |

## 1. Theory

### 1.1 Why tiny, high-speed objects are hard
| Difficulty | Effect on a standard detector |
|---|---|
| **Tiny size**: a tennis ball covers ≈ 0.001 % of a 1280×720 frame | Repeated down-sampling in CNN backbones (stride 16–32) erases the object completely. Anchor-based detectors (YOLO, Faster R-CNN) have no anchor that small. |
| **High speed**: 150–250 km/h smashes, i.e. > 30 px/frame at 30 fps | Motion blur turns the ball into a faint, semi-transparent streak whose appearance depends on its velocity. |
| **Occlusion**: by players, rackets and the net | The object is literally not in the frame for several frames; only *temporal context* can recover it. |
| **Clutter**: court lines, white shoes, crowd, logos | Many static blobs look exactly like the ball in a single frame. **Motion** is the key cue that tells them apart. |

Classical pipelines (background subtraction → blob filtering → Kalman / Viterbi track association) depend heavily on hand-tuned thresholds and break under camera motion and lighting changes.

### 1.2 The TrackNet family
The TrackNet idea is to treat ball localisation as **dense heatmap regression from a stack of consecutive frames**, so the network learns both *appearance* and *flight pattern*.

| Version | Year | Key idea |
|---|---|---|
| **TrackNet V1** (Huang et al., AVSS) | 2019 | VGG-16 encoder + DeconvNet decoder. Input: 3 RGB frames (9 ch) at 640×360. Output: a **256-class softmax per pixel** over Gaussian heatmap grey levels (σ²=10), trained with cross-entropy. Decoded by threshold 128 + Hough circle. Tennis: P/R/F1 = 99.7 / 97.3 / 98.5 %. |
| **TrackNetV2** (Sun et al.) | 2020 | **U-Net skip connections**, input reduced to 512×288, **MIMO** (3 frames in → 3 heatmaps out), a **sigmoid** output with **weighted BCE** (a focal-style loss), and a shuttlecock dataset. |
| **TrackNetV3** (Chen & Wang, ACM MM Asia) | 2023 | Background-estimate input, mixup augmentation, and a learned **trajectory rectification** (inpainting) network for occluded segments. |
| **TrackNetV4** (Raj et al., arXiv:2409.14543) | 2024 | **Motion attention maps**: absolute frame differences → learnable power normalisation → element-wise fusion with high-level features. A plug-and-play addition to V2/V3. |
| **TrackNetV5** (Tang et al., arXiv:2512.02789) | 2025 | **Motion Direction Decoupling** (signed polarity motion maps) + **R-STR** Transformer residual refinement. Reported F1 0.9859 vs V4 0.9581 / V2 0.9677, at +3.7 % FLOPs. **This is the paper we re-implement and modify.** |

### 1.3 TrackNetV5 in detail
**Backbone.** The V2 U-Net: a VGG-style encoder (64·2 / 128·2 / 256·3 / 512·3 convs with 3 max-pools) and a symmetric decoder with skip concatenation. MIMO: frames $(I_{t-1}, I_t, I_{t+1})$ map to three heatmaps.

**Motion Direction Decoupling (MDD).** V4 used $|I_{t+1}-I_t|$, which throws away the *sign* of the change. A ball entering a pixel (brightening) and leaving it (darkening) look identical, so the direction of motion is lost. V5 splits each frame difference into two polarity fields:
$$\Delta_{t-1\to t}=I_t-I_{t-1},\quad \Delta_{t\to t+1}=I_{t+1}-I_t,\qquad P^{+}(\Delta)=\mathrm{ReLU}(\Delta),\quad P^{-}(\Delta)=\mathrm{ReLU}(-\Delta)$$
Each field is mapped to an attention value by a learnable soft threshold:
$$A=\sigma\!\big(k(\alpha)\,(|x|-m(\beta))\big),\qquad k(\alpha)=\frac{5}{0.45\,|\tanh\alpha|+\varepsilon},\qquad m(\beta)=0.6\tanh\beta$$
Here $k$ is the slope and $m$ the motion threshold. The network input is the 13-channel tensor $[I_{t-1},A_{t-1,t},I_t,A_{t,t+1},I_{t+1}]$ (3 × 3 RGB + 2 × 2 polarity maps). The maps are reused in the prediction head to gate the output.

**Residual-Driven Spatio-Temporal Refinement (R-STR).** A $1\times1$ conv produces a *draft* heatmap, which is fused with the MDD maps to give $\text{Draft}_{MDD}$. A lightweight TimeSformer-style Transformer (**TSATTHead**) takes it and predicts a residual $\Delta$. The head has non-overlapping patch embedding, factorised spatial + temporal positional encodings, *divided* space/time attention, and PixelShuffle decoding.
$$\text{train: } H=\sigma(\text{Draft}'+\Delta),\ \text{Draft}'=\text{Dropout}_{0.1}(\text{Draft});\qquad \text{inference: } H=\sigma(\text{Draft}_{MDD}+\Delta)$$

**Loss & targets.** V5 trains with weighted binary cross-entropy (from V2), whose focal-style weights down-weight easy pixels:
$$\mathcal{L}_{WBCE}=-\frac1N\sum\Big[(1-p)^2\,y\log p+p^2\,(1-y)\log(1-p)\Big]$$
The target $y$ is a **binary disk of radius r = 30 px** (on the 1280-wide original) around the ball centre. Training uses AdamW (lr $10^{-4}$, batch 2, 30 epochs, step decay at 20/25).

### 1.4 Weaknesses we identified in V5 → our modifications
| ID | Weakness in V5 (as specified in the paper) | Our modification |
|---|---|---|
| **M1** | A 60-px-wide *binary disk* target is much larger than a 5–10 px ball. Every pixel in the disk is "equally ball", so the peak is flat. The decoded centre (bbox of a thresholded blob) is biased, and two nearby balls or ball + racket merge. | **Ball-size-aware Gaussian targets + sub-pixel soft-argmax decoding** |
| **M2** | Eq. 2 applies $|x|$ to fields that are *already* non-negative, with **one shared (α, β)** for both polarities and both frame pairs. Arrival (+) and departure (−) evidence are thresholded identically, and the *direction* is left implicit. | **Per-pair, per-polarity learnable (α, β) + an explicit direction channel** $A^{+}-A^{-}$ |
| **M3** | **Train/inference mismatch**: the head is trained on Dropout(Draft) but run on Draft$_{MDD}$, an input distribution it never saw. The full-resolution Transformer is also the most expensive part of the model. | **Consistent** Draft$_{MDD}$ in both phases + a **slim** TSATT head at ¼ the tokens |
| **M4** | 14.8 M parameters / 117 GFLOPs. Heavy for edge / CPU deployment in stadium cameras. | **Depthwise-separable** convolutions in all but the first stage |
| **M5** | The authors' stated limitation is *false positives in complex backgrounds*. There is also no explicit occlusion handling (V3's rectification is not part of V5). | **Physics-aware post-processing**: a constant-acceleration Kalman filter with χ² gating + quadratic (ballistic) gap filling |

### 1.5 Evaluation protocol (TrackNet standard)
For each frame, with prediction $\hat p$ and ground truth $p$ (when the ball is present):

- **TP**: ball present, predicted, $\|\hat p-p\|\le\tau$
- **FP1**: ball present, predicted, but $\|\hat p-p\|>\tau$
- **FP2**: ball absent, but predicted
- **FN**: ball present, not predicted
- **TN**: ball absent, not predicted

$\text{Precision}=\frac{TP}{TP+FP1+FP2}$, $\text{Recall}=\frac{TP}{TP+FN}$, $F1=\frac{2PR}{P+R}$, $\text{Acc}=\frac{TP+TN}{\text{all}}$.

V4/V5 use τ = 4 px at 1280×720. Our frames are 512 px wide (GPU profile) or 288 px wide (CPU profile), so τ = 4 px at 1280 would be only 1.6 px or 0.9 px here, which is close to the label quantisation. We use a headline **τ ≈ 13 px at 1280 scale** (roughly one ball diameter), i.e. **5.3 px at 512** or **3 px at 288**. All pixel constants scale with `SC = W/288`. We also report the and report the **full F1-vs-τ curve** so stricter tolerances can be compared as well.

### 1.4.1 How this work differs from existing approaches
| Approach | Input | Motion cue | Training target | Refinement | Post-processing | Occlusion handling |
|---|---|---|---|---|---|---|
| Classical (background subtraction + Kalman) | 1 frame + background model | frame differencing (hand-tuned) | — | — | track association | track coasting |
| TrackNet V1 (2019) | 3 frames, 640×360 | implicit (learned from stacked frames) | 256-class softmax over a Gaussian | — | threshold + Hough circle | none |
| TrackNetV2 (2020) | 3 frames, 512×288, MIMO | implicit | binary map, weighted BCE | — | blob centre | none |
| TrackNetV3 (2023) | 3 frames + background image | background difference | binary map + mixup | **learned trajectory rectification network** | — | inpainting network (extra model) |
| TrackNetV4 (2024) | 3 frames | **\|frame diff\|** → learnable power-norm attention | binary map | — | blob centre | none |
| TrackNetV5 (2025) | 3 frames + 4 polarity maps | **signed polarity (MDD)**, shared α, β | binary disk r = 30 px | **Transformer residual (R-STR)**, trained on a different input than at test time | blob centre | implicit only |
| **TrackNetV5-Lite+ (ours)** | 3 frames + 4 polarity + 2 direction maps | **per-pair / per-polarity MDD + explicit direction** (M2) | **ball-size Gaussian + unbiased sub-pixel decoding** (M1) | **consistent, 4× cheaper R-STR** (M3) on a **depthwise-separable** backbone (M4) | **Kalman χ² gating** (M5) | **ballistic gap filling, training-free** (M5) |

Measured comparisons (parameters, FLOPs, FPS, F1 on synthetic and on real footage) are produced in §10 (`figures/tab05_comparison.png`).

### 1.4.2 Novel contributions of this work
1. **M1. Ball-size-aware Gaussian targets with an unbiased sub-pixel decoder** (value-thresholded centroid). This replaces V5's 60-px binary disk, which is wider than the ball itself.
2. **M2. Polarity-specific motion decoupling.** It uses separate learnable slopes and thresholds per frame pair and per polarity, and adds an explicit signed direction channel. This fixes V5's shared threshold, which is applied after an `|x|` that has no effect.
3. **M3. A consistent and slim residual refinement head.** We identify and remove V5's train/inference input mismatch (Dropout(Draft) in training vs. Draft$_{MDD}$ at test time), and cut the head's tokens by 4×.
4. **M4. A depthwise-separable TrackNet backbone.** It gives 7× fewer parameters and 5× fewer FLOPs, and we measure the FLOPs-vs-latency gap on real hardware.
5. **M5. Physics-aware, training-free post-processing.** A constant-acceleration Kalman filter with χ² gating and ballistic gap filling targets V5's stated failure mode (false positives) and adds the occlusion recovery that V5 lacks.
6. **Evaluation contributions.**
   - A procedural broadcast generator with exact occlusion and speed labels, plus a **match-level** sim-to-real protocol on the TrackNet tennis dataset (fine-tune on games 1–8, test on games 9–10).
   - A 48-check test suite that validates every component.
   - A one-rung-at-a-time ablation that attributes each gain to a single change.

## 2. Setup & configuration
Everything is controlled by `CFG`. The profile is picked automatically, or set by hand with `TN_PROFILE`:

| Profile | When | Resolution | Width | Data (train clips × frames) | Steps × batch | Expected time |
|---|---|---|---|---|---|---|
| **`gpu`** *(default when CUDA is available, e.g. Kaggle T4 / P100)* | the main experiment | **512×288** (the paper's) | ½ | 150 × 60 | 6,000 × 8 (+ 3,000 × 8 fine-tuning per model on real data) | ≈ 3–3.5 h on a Kaggle T4, ≈ 5 h with the real dataset |
| `gpu_full` | full-width replication | 512×288 | 1 (paper's) | 240 × 60 | 15,000 × 8 (+ 6,000) | ≈ 10+ h |
| `cpu` *(default without CUDA)* | local debugging only | 288×160 | ¼ | 48 × 60 | 1,500 × 4 (+ 300) | ≈ 2.5–3 h on a laptop CPU |

Resolution and channel width are the **only** deviations from the papers' architectures in the faithful re-implementations.

Environment variables: `TN_PROFILE=gpu|gpu_full|cpu`, `TN_SMOKE=1` (a 1-minute end-to-end check), `TN_RETRAIN=1` (ignore cached checkpoints).

**Running on Kaggle:** create a notebook → *File → Import Notebook* → upload this `.ipynb`. In *Settings*, set **Accelerator = GPU T4 ×1 (or P100)**. Then use *Save Version → Save & Run All (Commit)*, which runs headless for up to 12 h. All dependencies are preinstalled on Kaggle. The last cell bundles `figures/`, `results/` and the checkpoints into `tracknet_outputs.zip` under *Output*.

In [ ]:
import os, re, time, math, json, random, copy, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.flop_counter import FlopCounterMode
from IPython.display import display, Markdown, Image as IPImage

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
T_START = time.time()                  # wall-clock of the whole run (reported in results/environment.md)
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROFILE = os.environ.get("TN_PROFILE", "gpu" if DEVICE.type == "cuda" else "cpu")
SMOKE   = os.environ.get("TN_SMOKE", "0") == "1"
RETRAIN = os.environ.get("TN_RETRAIN", "0") == "1"
IS_KAGGLE = Path("/kaggle/working").exists()
NUM_WORKERS = 0 if os.name == "nt" else min(4, os.cpu_count() or 1)   # Windows notebooks can't fork workers

PROFILES = {
    #        working res     backbone width   synthetic clips (train/val/test)          optimisation
    "gpu":      dict(W=512, H=288, width=0.50, clip_len=60, n_train=150, n_val=12, n_test=30,
                     steps=6000, batch=8, lr=1e-3, eval_every=750, ft_steps=3000, ft_lr=3e-4, ft_eval_every=500),
    "gpu_full": dict(W=512, H=288, width=1.00, clip_len=60, n_train=240, n_val=20, n_test=40,
                     steps=15000, batch=8, lr=5e-4, eval_every=1500, ft_steps=6000, ft_lr=1.5e-4, ft_eval_every=1000),
    "cpu":      dict(W=288, H=160, width=0.25, clip_len=60, n_train=48, n_val=6, n_test=14,
                     steps=1500, batch=4, lr=2e-3, eval_every=250, ft_steps=300, ft_lr=6e-4, ft_eval_every=100),
}
CFG = dict(PROFILES[PROFILE])
if SMOKE:
    CFG.update(n_train=3, n_val=1, n_test=2, steps=4, batch=2, eval_every=2, ft_steps=4, ft_eval_every=2)
CFG.update(
    tol=round(3.0 * CFG["W"] / 288, 2), # headline tolerance: 3 px at 288 wide == 5.33 px at 512 (≈13 px at 1280)
    disk_r=30 * CFG["W"] / 1280,      # V5's r=30 px disk target, rescaled to our resolution
    det_thr=0.5,                      # heatmap probability threshold for a detection
    real_train_games=list(range(1, 9)),   # real TrackNet tennis: fine-tune on broadcast games 1-8 ...
    real_test_games=[9, 10],              # ... and test on the unseen games 9-10 (game-level split)
)
SC = CFG["W"] / 288                    # pixel-scale factor vs. the 288-px CPU profile

ROOT = Path.cwd()
FIG, OUT, CKPT, RES = ROOT / "figures", ROOT / "outputs", ROOT / "checkpoints", ROOT / "results"
for d in (FIG, OUT, CKPT, RES):
    d.mkdir(exist_ok=True)

torch.set_num_threads(max(1, os.cpu_count() - 2)) if DEVICE.type == "cpu" else None
seed_everything()
print(f"torch {torch.__version__} | device={DEVICE} | profile={PROFILE} | smoke={SMOKE} | threads={torch.get_num_threads()}")
print(json.dumps(CFG, indent=1))

In [ ]:
# ---- Plot style: one consistent, colour-blind-validated palette for every figure ----
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e0"
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": MUTED, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "lines.linewidth": 2, "font.size": 9.5,
    "legend.frameon": False, "axes.prop_cycle": mpl.cycler(color=PAL), "figure.dpi": 110,
})

def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

def save_table(df, name, title, fmt="{:.3f}", col_width=1.15, first_col_width=2.6):
    """Render a DataFrame as a PNG (for the report) and save it next to the figures."""
    d = df.copy()
    rows = [" · ".join(map(str, i)) if isinstance(i, tuple) else str(i) for i in d.index]
    def f(v):
        if isinstance(v, (float, np.floating)):
            return "—" if np.isnan(v) else fmt.format(v)
        return str(v)
    cells = [[f(v) for v in r] for r in d.values]
    nr, nc = len(rows), len(d.columns)
    fig, ax = plt.subplots(figsize=(first_col_width + col_width * nc, 0.55 + 0.34 * (nr + 1)))
    ax.axis("off")
    tb = ax.table(cellText=cells, rowLabels=rows, colLabels=[str(c) for c in d.columns], loc="center", cellLoc="center")
    tb.auto_set_font_size(False); tb.set_fontsize(8.5); tb.scale(1, 1.35)
    tb.auto_set_column_width(col=list(range(-1, nc)))          # fit every column to its text
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor(GRID)
        if r == 0 or c == -1:
            cell.set_text_props(weight="bold", color=INK); cell.set_facecolor("#efeeea")
    ax.set_title(title, loc="left", fontsize=10.5, pad=10)
    savefig(fig, name)


def no_axes(ax):
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

### 1.6 Visual summary of the TrackNet lineage and of the V5 pipeline

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.6))
no_axes(ax)
events = [(2019, "TrackNet V1", "VGG+DeconvNet\n256-class heatmap"),
          (2020, "TrackNetV2", "U-Net · MIMO\nweighted BCE"),
          (2023, "TrackNetV3", "background input\ntrajectory rectification"),
          (2024, "TrackNetV4", "motion attention\n|frame diff|"),
          (2025, "TrackNetV5", "MDD polarity +\nR-STR Transformer"),
          (2026, "V5-Lite+ (ours)", "M1–M5:\naccuracy & efficiency")]
ax.plot([2018.6, 2026.4], [0, 0], color=MUTED, lw=1.5)
for i, (yr, name, desc) in enumerate(events):
    c = PAL[1] if "ours" in name else (PAL[0] if yr >= 2024 else MUTED)
    ax.scatter([yr], [0], s=90, color=c, zorder=3, edgecolor="#fcfcfb", linewidth=2)
    if i % 2 == 0:
        ax.text(yr, 0.2, desc, ha="center", va="bottom", fontsize=8, color=INK2)
        ax.text(yr, 0.75, name, ha="center", va="bottom", fontweight="bold", fontsize=9.5)
    else:
        ax.text(yr, -0.2, name, ha="center", va="top", fontweight="bold", fontsize=9.5)
        ax.text(yr, -0.55, desc, ha="center", va="top", fontsize=8, color=INK2)
ax.set_ylim(-1.4, 1.4); ax.set_xlim(2018.4, 2026.6)
ax.set_title("Evolution of heatmap-based tiny-object tracking (TrackNet family)", loc="left")
savefig(fig, "fig01_tracknet_timeline")

In [ ]:
def draw_pipeline(ax, boxes, arrows, title):
    no_axes(ax); ax.set_xlim(0, 100); ax.set_ylim(0, 30); ax.set_title(title, loc="left")
    for (x, y, w, h, txt, col) in boxes:
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.4,rounding_size=1.2",
                                    fc=col + "22", ec=col, lw=1.5))
        ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=8)
    for (x0, y0, x1, y1) in arrows:
        ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1.2))

fig, axes = plt.subplots(2, 1, figsize=(12, 5.4))
B = PAL[0]; O = PAL[1]; G = PAL[2]; V = PAL[6]
draw_pipeline(axes[0], [
    (1, 10, 11, 10, "3 RGB frames\n$I_{t-1},I_t,I_{t+1}$", MUTED),
    (16, 18, 15, 9, "MDD\n$P^\\pm$ → σ(k(|x|−m))\nshared α, β", B),
    (16, 2, 15, 9, "concat → 13 ch", MUTED),
    (36, 6, 16, 14, "U-Net backbone\n(V2, 3×3 convs)\n11 M params", MUTED),
    (57, 6, 13, 14, "draft heatmaps\n(1×1 conv)\n+ MDD fusion", V),
    (75, 6, 13, 14, "TSATTHead\npatch 8, d=64, L=2\nspace / time attn", V),
    (91, 9, 8, 8, "σ(draft+Δ)\nbinary disk\nr=30", G)],
    [(12, 15, 16, 22), (12, 15, 16, 6), (23.5, 18, 23.5, 11), (31, 6.5, 36, 12), (52, 13, 57, 13), (70, 13, 75, 13), (88, 13, 91, 13)],
    "TrackNetV5 (faithful, as described in the paper)")
draw_pipeline(axes[1], [
    (1, 10, 11, 10, "3 RGB frames", MUTED),
    (16, 18, 15, 9, "MDD-M2\nper-pair/polarity (α,β)\n+ direction $A^+−A^-$", O),
    (16, 2, 15, 9, "concat → 15 ch", MUTED),
    (36, 6, 16, 14, "U-Net backbone\nM4: depthwise-\nseparable convs", O),
    (57, 6, 13, 14, "draft + MDD\nM3: same input\nin train & test", O),
    (75, 6, 13, 14, "slim TSATT (M3)\npatch 16, d=32, L=1\n¼ tokens", O),
    (91, 9, 8, 8, "M1: Gaussian\nsoft-argmax\nM5: Kalman", O)],
    [(12, 15, 16, 22), (12, 15, 16, 6), (23.5, 18, 23.5, 11), (31, 6.5, 36, 12), (52, 13, 57, 13), (70, 13, 75, 13), (88, 13, 91, 13)],
    "TrackNetV5-Lite+ (ours): changed components in orange")
plt.tight_layout()
savefig(fig, "fig02_pipeline_v5_vs_ours")

## 3. Synthetic broadcast-video generator
The real datasets are several GB, and training on them at full resolution needs a GPU. So we built a **procedural renderer** that reproduces the *statistical difficulties* of racket-sport broadcasts and provides exact ground truth:

* **Perspective court**: a trapezoid court with lines and a net, three surface types (hard / clay / grass), smooth illumination fields and texture noise. A **crowd band** contains static bright specks that act as ball-like distractors.
* **Physics**: the ball moves on the court plane, with height following $z(t)=z_0+v_zt-\tfrac12gt^2$. It bounces with restitution $e\in[0.55,0.75]$ and is hit back and forth between two players. Projection uses a perspective scale $s(w)$, so far balls are smaller and slower in pixels.
* **Tiny ball**: radius ≈ 1.2–3.2 px depending on depth (≈ 5–14 px at 1280×720, matching real broadcasts).
* **Motion blur**: the ball is integrated over a half-frame exposure along its image velocity, producing faint streaks at high speed.
* **Players**: they chase the ball, are drawn in depth order, and **occlude** it. Occluded frames are labelled `vis=2` with the true position, as in TrackNet's labelling convention.
* **Dead-ball segments**: in 60 % of clips the ball is absent for 6–14 frames (between points). This measures **false positives** on look-alike distractors, which V5's authors name as their main failure mode.
* **Camera**: sub-pixel jitter, brightness flicker, sensor noise, slight defocus.

Labels per frame: `x, y, vis (0 = out of frame, 1 = visible, 2 = occluded), speed (px/frame), radius (px), near_player`.

In [ ]:
SURFACES = [  # (court RGB, surround RGB, line RGB)  values in [0, 1]
    ((0.23, 0.42, 0.62), (0.20, 0.47, 0.33), (0.95, 0.95, 0.95)),   # blue hard court
    ((0.72, 0.38, 0.22), (0.64, 0.33, 0.20), (0.96, 0.94, 0.90)),   # clay
    ((0.33, 0.55, 0.25), (0.28, 0.48, 0.22), (0.93, 0.95, 0.93)),   # grass
    ((0.25, 0.48, 0.38), (0.36, 0.27, 0.42), (0.95, 0.95, 0.95)),   # green hard court
]

def smooth_noise(rng, H, W, scale, amp):
    small = rng.standard_normal((max(2, H // scale), max(2, W // scale))).astype(np.float32)
    return cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC) * amp


class Court:
    """Static background + perspective mapping from court coords (u lateral, w depth in [0,1]) to pixels."""
    def __init__(self, rng, W, H):
        self.W, self.H = W, H
        court, surround, line = SURFACES[rng.integers(len(SURFACES))]
        self.y_far, self.y_near = H * rng.uniform(0.26, 0.32), H * rng.uniform(0.90, 0.97)
        self.w_far, self.w_near = W * rng.uniform(0.40, 0.50), W * rng.uniform(0.80, 0.92)
        self.cx = W / 2 + rng.uniform(-0.04, 0.04) * W
        bg = np.empty((H, W, 3), np.float32); bg[:] = surround
        corners = np.array([self.to_px(u, w, 0)[:2] for u, w in [(0, 0), (1, 0), (1, 1), (0, 1)]], np.float32)
        cv2.fillPoly(bg, [np.round(corners * 16).astype(np.int32)], court, lineType=cv2.LINE_AA, shift=4)
        # court lines: outline, singles sidelines, service lines, centre line
        segs = [((0, 0), (1, 0)), ((1, 0), (1, 1)), ((1, 1), (0, 1)), ((0, 1), (0, 0)),
                ((0.12, 0), (0.12, 1)), ((0.88, 0), (0.88, 1)), ((0.12, 0.27), (0.88, 0.27)),
                ((0.12, 0.73), (0.88, 0.73)), ((0.5, 0.27), (0.5, 0.73))]
        for (a, b) in segs:
            p0 = np.round(np.array(self.to_px(*a, 0)[:2]) * 16).astype(int)
            p1 = np.round(np.array(self.to_px(*b, 0)[:2]) * 16).astype(int)
            cv2.line(bg, tuple(p0), tuple(p1), line, 1, cv2.LINE_AA, shift=4)
        # net (at depth 0.5) with white tape
        nl, nr = self.to_px(-0.05, 0.5, 0), self.to_px(1.05, 0.5, 0)
        net_h = 9 * self.scale(0.5) * H / 160
        net_y = int(round(nl[1]))
        cv2.rectangle(bg, (int(nl[0]), int(net_y - net_h)), (int(nr[0]), net_y), (0.12, 0.12, 0.12), -1)
        cv2.line(bg, (int(nl[0]), int(net_y - net_h)), (int(nr[0]), int(net_y - net_h)), (0.95, 0.95, 0.95), 1)
        # crowd band with static ball-like distractor specks
        top = int(self.y_far - H * 0.05)
        crowd = rng.uniform(0.1, 0.6, (max(1, top // 2), W // 2, 3)).astype(np.float32)
        bg[:top] = cv2.resize(crowd, (W, top), interpolation=cv2.INTER_NEAREST) * 0.8 + 0.1
        for _ in range(rng.integers(6, 18)):
            col = (0.95, 0.95, rng.uniform(0.3, 0.95))
            cv2.circle(bg, (int(rng.uniform(0, W) * 16), int(rng.uniform(0, top) * 16)),
                       int(rng.uniform(1.2, 2.5) * 16), col, -1, cv2.LINE_AA, shift=4)
        # illumination + texture
        bg *= (1 + smooth_noise(rng, H, W, 24, 0.08))[..., None]
        bg += rng.normal(0, 0.012, bg.shape).astype(np.float32)
        self.bg = np.clip(bg, 0, 1)

    def scale(self, w):
        return 0.55 + 0.45 * w

    def to_px(self, u, w, z):
        width = self.w_far + w * (self.w_near - self.w_far)
        x = self.cx + (u - 0.5) * width
        y = self.y_far + w * (self.y_near - self.y_far) - z * self.scale(w) * self.H / 160
        return x, y, self.scale(w)


def draw_player(canvas, mask, x, y, s, shirt, H):
    """Simple player sprite (legs, body, head, racket) anchored at feet (x, y); s = perspective scale."""
    k = 1.3 * s * H / 160
    body_h, body_w = 18 * k, 5.5 * k
    S = 16
    def P(px, py): return (int(round(px * S)), int(round(py * S)))
    for img, col in ((canvas, None), (mask, 1.0)):
        c_body = col if col is not None else shirt
        c_skin = col if col is not None else (0.85, 0.65, 0.5)
        c_leg = col if col is not None else (0.15, 0.15, 0.18)
        cv2.line(img, P(x - 2 * k, y), P(x - 1 * k, y - 8 * k), c_leg, max(1, int(2 * k)), cv2.LINE_AA, shift=4)
        cv2.line(img, P(x + 2 * k, y), P(x + 1 * k, y - 8 * k), c_leg, max(1, int(2 * k)), cv2.LINE_AA, shift=4)
        cv2.ellipse(img, P(x, y - 8 * k - body_h * 0.3), P(body_w / 2, body_h * 0.32), 0, 0, 360, c_body, -1, cv2.LINE_AA, shift=4)
        cv2.circle(img, P(x, y - 8 * k - body_h * 0.68), int(2.3 * k * S), c_skin, -1, cv2.LINE_AA, shift=4)
        cv2.line(img, P(x + body_w / 2, y - 12 * k), P(x + 7 * k, y - 17 * k), c_leg, 1, cv2.LINE_AA, shift=4)
        cv2.ellipse(img, P(x + 8 * k, y - 19 * k), P(1.6 * k, 2.4 * k), 30, 0, 360, c_body if col else (0.9, 0.9, 0.9), 1, cv2.LINE_AA, shift=4)

In [ ]:
def simulate_rally(rng, L):
    """Ball state in court coordinates (u, w, z) for L frames: alternating shots, gravity, bounces."""
    g = rng.uniform(0.35, 0.65)            # gravity in 'court pixels'/frame^2
    e = rng.uniform(0.55, 0.75)            # restitution
    u, w, z = rng.uniform(0.2, 0.8), rng.choice([0.02, 0.98]), rng.uniform(4, 9)
    traj = []
    def new_shot(u, w, z):
        toward_far = w > 0.5
        T = rng.uniform(9, 26)                          # frames to bounce (smaller = faster shot)
        tu, tw = rng.uniform(0.05, 0.95), (rng.uniform(0.05, 0.4) if toward_far else rng.uniform(0.6, 0.95))
        du, dw = (tu - u) / T, (tw - w) / T
        vz = (0.5 * g * T ** 2 - z) / T                # lands at t = T
        return du, dw, vz
    du, dw, vz = new_shot(u, w, z)
    for t in range(L):
        traj.append((u, w, z))
        u, w = u + du, w + dw
        vz -= g; z += vz
        if z < 0:                                         # bounce
            z, vz = -z * e, -vz * e
            du, dw = du * 0.85, dw * 0.85
        heading_far, heading_near = dw < 0, dw > 0
        if (heading_far and w < 0.03) or (heading_near and w > 0.97) or (z < 1.5 and (w < -0.05 or w > 1.05)):
            z = max(z, rng.uniform(3, 9))                 # player hits it back
            du, dw, vz = new_shot(u, float(np.clip(w, 0, 1)), z)
            if rng.random() < 0.15:                        # occasional lob
                vz += rng.uniform(2, 4)
    return np.array(traj, np.float32)


def render_clip(seed, L, W, H, ball_color=None):
    """ball_color overrides the ball's RGB (used by the unit tests to locate ball pixels); RNG draws are unchanged."""
    rng = np.random.default_rng(seed)
    court = Court(rng, W, H)
    traj = simulate_rally(rng, L)
    ball_col = np.array((0.90, 0.95, 0.25) if rng.random() < 0.75 else (0.97, 0.97, 0.97), np.float32)
    if ball_color is not None:
        ball_col = np.array(ball_color, np.float32)
    r0 = rng.uniform(2.3, 3.2) * H / 160
    shirts = [tuple(rng.uniform(0.1, 0.95, 3)) for _ in range(2)]
    players = [[rng.uniform(0.3, 0.7), -0.02, rng.normal(0, 0.02)], [rng.uniform(0.3, 0.7), 1.0, rng.normal(0, 0.02)]]   # far, near (u, w, offset)
    noise_sd = rng.uniform(0.004, 0.02)
    jitter = np.cumsum(rng.normal(0, 0.25, (L, 2)), 0).clip(-3, 3) * H / 160     # camera shake (moves the whole frame)
    dead = np.zeros(L, bool)                       # dead-ball segment: no ball in frame (tests false positives)
    if rng.random() < 0.6:
        d0 = rng.integers(0, L - 6); dead[d0:d0 + rng.integers(6, 15)] = True
    frames = np.zeros((L, H, W, 3), np.uint8)
    labels = np.zeros((L, 6), np.float32)         # x, y, vis, speed, radius, near_player
    for t in range(L):
        u, w, z = traj[t]
        # players chase the ball laterally when it is on their half
        for p in players:
            if (p[1] < 0.5) == (w < 0.5):               # ball on my half: run to intercept it
                p[0] += np.clip(u + p[2] - p[0], -0.06, 0.06)
            p[0] += rng.normal(0, 0.004)
        x, y, s = court.to_px(u, w, z)                 # scene coordinates (before camera shake)
        r = r0 * s
        # motion blur: integrate along the image-space path over half a frame of exposure
        if t > 0:
            pu, pw, pz = traj[t - 1]
            px_, py_, _ = court.to_px(pu, pw, pz)
        else:
            px_, py_ = x, y
        speed = math.hypot(x - px_, y - py_)
        n_sub = int(np.clip(math.ceil(0.5 * speed / max(r, 0.5) * 2), 1, 14))
        alpha = np.zeros((H, W), np.float32)
        for k in range(n_sub):
            f = 0.5 * k / max(n_sub - 1, 1) if n_sub > 1 else 0.0
            bx, by = x - f * (x - px_), y - f * (y - py_)
            tmp = np.zeros((H, W), np.float32)
            cv2.circle(tmp, (int(round(bx * 16)), int(round(by * 16))), max(1, int(round(r * 16))), 1.0, -1, cv2.LINE_AA, shift=4)
            alpha += tmp
        alpha = np.clip(alpha / n_sub * min(1.0, 0.55 + 0.45 * n_sub / 2), 0, 1)
        if r < 1.3:                         # sub-pixel balls cannot reach full contrast
            alpha *= 0.85
        if dead[t]:
            alpha[:] = 0
        # composite in depth order: players behind ball -> ball -> players in front
        img = court.bg.copy()
        img = np.clip(img * (1 + rng.normal(0, 0.015)), 0, 1)
        front_mask = np.zeros((H, W), np.float32)
        player_px = []
        order = sorted(range(2), key=lambda i: players[i][1])
        ball_drawn = False
        for i in order:
            pu_, pw_, _ = players[i]
            if pw_ > w and not ball_drawn:
                img = img * (1 - alpha[..., None]) + ball_col * alpha[..., None]; ball_drawn = True
            fx, fy, fs = court.to_px(pu_, pw_, 0)
            m = np.zeros((H, W), np.float32)
            draw_player(img, m, fx, fy, fs, shirts[i], H)
            player_px.append((fx, fy, fs))
            if ball_drawn:
                front_mask = np.maximum(front_mask, m)
        if not ball_drawn:
            img = img * (1 - alpha[..., None]) + ball_col * alpha[..., None]
        # camera: sub-pixel shake of the whole frame (court, players, ball), sensor noise, slight defocus
        jx, jy = jitter[t]
        img = cv2.warpAffine(img, np.float32([[1, 0, jx], [0, 1, jy]]), (W, H), flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REFLECT101)
        img = img + rng.normal(0, noise_sd, img.shape).astype(np.float32)
        if rng.random() < 0.3:
            img = cv2.GaussianBlur(img, (3, 3), 0.6)
        frames[t] = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        xi, yi = x + jx, y + jy                          # label = position in the shaken image
        inside = (0 <= xi < W) and (0 <= yi < H)
        head = np.zeros((H, W), np.float32)              # the ball itself at the labelled position (not its blur tail)
        cv2.circle(head, (int(round(x * 16)), int(round(y * 16))), max(1, int(round(r * 16))), 1.0, -1, cv2.LINE_AA, shift=4)
        occluded = float((head * front_mask).sum() / max(head.sum(), 1e-6)) > 0.5   # player hides most of the ball
        vis = 0 if (not inside or dead[t]) else (2 if occluded else 1)
        near = min((math.hypot(x - fx, y - (fy - 14 * fs * H / 160)) for fx, fy, fs in player_px)) < 14 * H / 160
        labels[t] = (xi, yi, vis, speed if t > 0 else 0.0, r, float(near))
    return frames, labels


def make_split(n_clips, seed0, L, W, H):
    out = [render_clip(seed0 + i, L, W, H) for i in range(n_clips)]
    return dict(frames=np.stack([o[0] for o in out]), labels=np.stack([o[1] for o in out]))

t0 = time.time()
W, H, L = CFG["W"], CFG["H"], CFG["clip_len"]
DATA = {
    "train": make_split(CFG["n_train"], 1_000, L, W, H),
    "val":   make_split(CFG["n_val"],   50_000, L, W, H),
    "test":  make_split(CFG["n_test"],  90_000, L, W, H),
}
for k, v in DATA.items():
    lab = v["labels"]
    print(f"{k:5s}: {lab.shape[0]:3d} clips × {lab.shape[1]} frames | visible {np.mean(lab[...,2]==1):.1%} "
          f"occluded {np.mean(lab[...,2]==2):.1%} out-of-frame {np.mean(lab[...,2]==0):.1%} | {v['frames'].nbytes/1e6:.0f} MB")
print(f"rendered in {time.time()-t0:.1f}s")

### 3.1 What the data looks like
Each row is one clip; the zoomed insets show how small and faint the ball really is. Green ring = visible, orange dashed = occluded (label only).

In [ ]:
def ball_inset(ax, frame, xy, size=18, scale=6):
    x, y = int(round(xy[0])), int(round(xy[1]))
    h, w = frame.shape[:2]
    x0, y0 = np.clip(x - size, 0, w - 2 * size), np.clip(y - size, 0, h - 2 * size)
    crop = frame[y0:y0 + 2 * size, x0:x0 + 2 * size]
    return cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_NEAREST), (x0, y0)

clips = [0, 1, 2, 3] if not SMOKE else [0, 1, 2]
fig, axes = plt.subplots(len(clips), 4, figsize=(13, 2.2 * len(clips)),
                         gridspec_kw=dict(width_ratios=[2.2, 2.2, 2.2, 1.2]))
for r, c in enumerate(clips):
    fr, lab = DATA["train"]["frames"][c % len(DATA["train"]["frames"])], DATA["train"]["labels"][c % len(DATA["train"]["labels"])]
    vis_idx = np.where(lab[:, 2] > 0)[0]
    ts = [vis_idx[len(vis_idx) // 5], vis_idx[len(vis_idx) // 2], vis_idx[4 * len(vis_idx) // 5]]
    for j, t in enumerate(ts):
        ax = axes[r, j]; no_axes(ax); ax.imshow(fr[t])
        ls = "-" if lab[t, 2] == 1 else "--"
        ax.add_patch(Circle(lab[t, :2], 7, fill=False, ec=PAL[2] if lab[t, 2] == 1 else PAL[1], lw=1.4, ls=ls))
        ax.set_title(f"clip {c} · t={t} · v={lab[t,3]:.1f}px/f · r={lab[t,4]:.1f}px", fontsize=8, loc="left")
    ins, _ = ball_inset(axes[r, 3], fr[ts[1]], lab[ts[1], :2])
    no_axes(axes[r, 3]); axes[r, 3].imshow(ins); axes[r, 3].set_title("6× zoom of middle frame", fontsize=8, loc="left")
fig.suptitle("Synthetic broadcast frames: tiny, blurred, occluded ball among static look-alike distractors", x=0.01, ha="left", fontweight="bold")
plt.tight_layout()
savefig(fig, "fig03_dataset_samples")

In [ ]:
lab_all = np.concatenate([DATA[s]["labels"].reshape(-1, 6) for s in ("train", "val", "test")])
pres = lab_all[:, 2] > 0
fig, axes = plt.subplots(1, 4, figsize=(14, 3.0))
axes[0].hist(lab_all[pres, 3], bins=30, color=PAL[0], edgecolor="#fcfcfb", linewidth=1)
axes[0].set(title="Ball speed", xlabel="px / frame", ylabel="frames")
axes[1].hist(lab_all[pres, 4], bins=30, color=PAL[0], edgecolor="#fcfcfb", linewidth=1)
axes[1].set(title="Ball radius", xlabel="px (at working res.)")
cats = ["out of frame", "visible", "occluded"]
cnt = [np.sum(lab_all[:, 2] == v) for v in (0, 1, 2)]
axes[2].bar(cats, cnt, color=[MUTED, PAL[0], PAL[1]], width=0.6)
for i, v in enumerate(cnt):
    axes[2].text(i, v, f"{v/len(lab_all):.1%}", ha="center", va="bottom", color=INK2)
axes[2].set(title="Visibility labels")
c0 = DATA["train"]["labels"][0]
for i in range(min(3, len(DATA["train"]["labels"]))):
    c0 = DATA["train"]["labels"][i]; m = c0[:, 2] > 0
    axes[3].plot(c0[m, 0], c0[m, 1], ".-", ms=3, lw=1, label=f"clip {i}")
axes[3].set(title="Example trajectories (image plane)", xlim=(0, W), ylim=(H, 0), xlabel="x", ylabel="y")
axes[3].legend(loc="lower right", fontsize=8)
plt.tight_layout()
savefig(fig, "fig04_dataset_stats")
print(f"speed: median {np.median(lab_all[pres,3]):.1f}, 95th pct {np.percentile(lab_all[pres,3],95):.1f} px/f | "
      f"radius: median {np.median(lab_all[pres,4]):.2f}px | near a player: {lab_all[pres,5].mean():.1%}")

### 3.2 Training targets: V5's binary disk vs. our ball-size-aware Gaussian (M1)
V5 uses a binary disk of radius r = 30 px on 1280-px frames (rescaled to **r ≈ 6.75 px** here). The ball itself has a radius of only 1.2–3.2 px. Inside the disk all pixels are equally "ball", so the network has **no incentive to put its maximum at the true centre**. M1 replaces the disk with a Gaussian whose σ follows the ball radius ($\sigma=\max(1,0.8\,r)$). The heatmap then encodes *where exactly* the centre is, and a soft-argmax can recover sub-pixel positions.

In [ ]:
YY, XX = np.mgrid[0:H, 0:W].astype(np.float32)

def make_heatmap(lab, kind):
    """lab: [T,6] -> [T,H,W] target. vis>0 (visible or occluded) frames get a target."""
    out = np.zeros((len(lab), H, W), np.float32)
    for i, (x, y, vis, _, r, _) in enumerate(lab):
        if vis == 0:
            continue
        sig = max(1.0, 0.8 * r)
        R = int(math.ceil(CFG["disk_r"] if kind == "disk" else 4 * sig)) + 1      # only touch a local window
        x0, x1, y0, y1 = max(0, int(x) - R), min(W, int(x) + R + 1), max(0, int(y) - R), min(H, int(y) + R + 1)
        if x0 >= x1 or y0 >= y1:
            continue
        d2 = (XX[y0:y1, x0:x1] - x) ** 2 + (YY[y0:y1, x0:x1] - y) ** 2
        out[i, y0:y1, x0:x1] = (d2 <= CFG["disk_r"] ** 2) if kind == "disk" else np.exp(-d2 / (2 * sig ** 2))
    return out

lab = DATA["train"]["labels"][0]
t = int(np.where(lab[:, 2] == 1)[0][len(np.where(lab[:, 2] == 1)[0]) // 2])
fr = DATA["train"]["frames"][0][t]
x, y = lab[t, :2]
x0, y0 = int(np.clip(x - 16, 0, W - 32)), int(np.clip(y - 16, 0, H - 32))
fig, axes = plt.subplots(1, 4, figsize=(12, 3.1))
axes[0].imshow(fr[y0:y0 + 32, x0:x0 + 32], interpolation="nearest"); axes[0].set_title("frame (32×32 crop)")
for ax, kind, ttl in [(axes[1], "disk", f"V5 target: disk r={CFG['disk_r']:.1f}px"), (axes[2], "gauss", "M1 target: Gaussian σ=0.8·r")]:
    hm = make_heatmap(lab[t:t + 1], kind)[0]
    ax.imshow(hm[y0:y0 + 32, x0:x0 + 32], cmap="Blues", vmin=0, vmax=1, interpolation="nearest"); ax.set_title(ttl)
for ax in axes[:3]:
    no_axes(ax); ax.scatter([x - x0], [y - y0], marker="+", s=80, color=PAL[1], lw=1.5)
row = int(round(y))
axes[3].plot(np.arange(x0, x0 + 32), make_heatmap(lab[t:t + 1], "disk")[0][row, x0:x0 + 32], label="disk (V5)", color=MUTED)
axes[3].plot(np.arange(x0, x0 + 32), make_heatmap(lab[t:t + 1], "gauss")[0][row, x0:x0 + 32], label="Gaussian (M1)", color=PAL[0])
axes[3].axvline(x, color=PAL[1], lw=1, ls="--"); axes[3].text(x + 0.5, 1.02, "true centre", color=INK2, fontsize=8)
axes[3].set(title="Horizontal profile through the ball", xlabel="x (px)", ylim=(0, 1.12)); axes[3].legend(loc="center right")
plt.tight_layout()
savefig(fig, "fig05_targets_disk_vs_gaussian")

In [ ]:
class WindowDataset(Dataset):
    """3-frame windows -> (9-ch input, 3 target heatmaps, 3 label rows).  MIMO like TrackNetV2/V5."""
    def __init__(self, split, target="disk", augment=False, stride=1):
        self.f, self.l = split["frames"], split["labels"]
        self.target, self.augment = target, augment
        self.idx = [(c, s) for c in range(len(self.l)) for s in range(0, len(self.l[c]) - 2, stride)]   # clips may differ in length

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        c, s = self.idx[i]
        fr = self.f[c][s:s + 3].astype(np.float32) / 255.0           # [3,H,W,3]
        lab = self.l[c][s:s + 3].copy()
        if self.augment and random.random() < 0.5:                    # horizontal flip
            fr = fr[:, :, ::-1]
            lab[:, 0] = (W - 1) - lab[:, 0]
        if self.augment:                                              # global brightness / contrast
            fr = np.clip((fr - 0.5) * random.uniform(0.85, 1.15) + 0.5 + random.uniform(-0.05, 0.05), 0, 1)
        x = torch.from_numpy(np.ascontiguousarray(fr.transpose(0, 3, 1, 2).reshape(9, H, W)))
        y = torch.from_numpy(make_heatmap(lab, self.target))
        return x, y, torch.from_numpy(lab)

## 4. Real TrackNet tennis dataset (game-level split)
The original TrackNet tennis dataset (Huang et al., 2019) has **10 broadcast matches, 95 rally clips and 19,835 labelled frames** at 1280×720. `Label.csv` columns: `file name, visibility, x-coordinate, y-coordinate, status`. Visibility codes: 0 = no ball, 1 = easily identifiable, 2 = hard to identify (blur, small), 3 = occluded.

**Split.** We split by **match**, not by frame, so that test courts, players, broadcasters and lighting are never seen in training. Random frame splits leak almost-identical neighbouring frames into the test set. TrackNetV4 reports this game-level protocol as the harder one.

| Split | Matches | Use |
|---|---|---|
| fine-tune | games 1–8, all clips except the last clip of each game | training |
| validation | last clip of each of games 1–8 | checkpoint selection only |
| **test** | **games 9–10** | reported numbers (never touched before evaluation) |

**Where the data is looked for:** `TN_REAL_ROOT`, then `data/tennis/`, then `Input/`, then any dataset attached on Kaggle (`/kaggle/input`). If none is found, the real-data parts (§9.5) are skipped and the notebook runs on synthetic data only. Frames are resized to the working resolution at load time (≈ 8.8 GB of RAM for all 19.8k frames at 512×288; Kaggle provides 29 GB).

In [ ]:
def find_real_root():
    """TN_REAL_ROOT, else data/tennis/ or Input/, else any attached Kaggle dataset containing TrackNet Label.csv files."""
    for cand in [os.environ.get("TN_REAL_ROOT"), ROOT / "data" / "tennis", ROOT / "Input", "/kaggle/input" if IS_KAGGLE else None]:
        if cand and Path(cand).exists() and next(Path(cand).rglob("Label.csv"), None) is not None:
            return Path(cand)
    return None


REAL_BALL_R = 4.0 * W / 1280           # a broadcast tennis ball is ~8 px wide at 1280x720 (sets the M1 Gaussian width)


def _num(pattern, path):
    """Number in the first path component matching e.g. 'game(\\d+)' (searched from the clip upwards); 0 if absent."""
    for part in reversed(path.parts):
        m = re.fullmatch(pattern, part, flags=re.I)
        if m:
            return int(m.group(1))
    return 0


def load_tennis_dataset(root, games=None, max_clips=None):
    """Loads TrackNet-V1-format clips (optionally only some games), resized to the working resolution.

    Returns the same structure as the synthetic splits (frames / labels per clip; label rows
    x, y, vis, speed, radius, 0) plus the original TrackNet visibility code per frame and the clip names.
    """
    clips = sorted(root.rglob("Label.csv"), key=lambda p: (_num(r"game(\d+)", p.parent), _num(r"clip(\d+)", p.parent)))
    if games is not None:
        clips = [p for p in clips if _num(r"game(\d+)", p.parent) in set(games)]
    frames, labels, vis_raw, names = [], [], [], []
    for csv in clips[:max_clips]:
        df = pd.read_csv(csv)
        df.columns = [c.strip().lower() for c in df.columns]
        imgs, labs, raws = [], [], []
        for _, row in df.iterrows():
            img = cv2.imread(str(csv.parent / str(row["file name"]).strip()))
            if img is None:
                continue
            h0, w0 = img.shape[:2]
            imgs.append(cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (W, H), interpolation=cv2.INTER_AREA))
            vis = int(row["visibility"]) if not pd.isna(row["visibility"]) else 0
            xx = float(row["x-coordinate"]) * W / w0 if vis > 0 and not pd.isna(row["x-coordinate"]) else 0
            yy = float(row["y-coordinate"]) * H / h0 if vis > 0 and not pd.isna(row["y-coordinate"]) else 0
            labs.append([xx, yy, 0 if vis == 0 else (2 if vis == 3 else 1), 0, REAL_BALL_R, 0])
            raws.append(vis)
        n = (len(imgs) // 3) * 3
        if n < 3:
            continue
        lab = np.array(labs[:n], np.float32)
        both = (lab[1:, 2] > 0) & (lab[:-1, 2] > 0)                 # image-space speed between labelled frames
        lab[1:, 3] = np.where(both, np.hypot(*(lab[1:, :2] - lab[:-1, :2]).T), 0)
        frames.append(np.stack(imgs[:n])); labels.append(lab); vis_raw.append(np.array(raws[:n]))
        game = _num(r"game(\d+)", csv.parent)               # (kept out of the f-string: Python < 3.12 on Kaggle)
        names.append(f"game{game}/{csv.parent.name}")
    return dict(frames=frames, labels=labels, vis_raw=vis_raw, clips=names) if frames else None


def split_real(root):
    """Game-level split: fine-tune on games 1-8 (last clip of each game held out for validation), test on 9-10."""
    tr = load_tennis_dataset(root, CFG["real_train_games"], max_clips=4 if SMOKE else None)
    te = load_tennis_dataset(root, CFG["real_test_games"], max_clips=2 if SMOKE else None)
    if not tr or not te:
        return None
    by_game = {}
    for i, name in enumerate(tr["clips"]):
        by_game.setdefault(name.split("/")[0], []).append(i)
    val_ids = [ids[-1] for ids in by_game.values() if len(ids) >= 2]
    tr_ids = [i for i in range(len(tr["clips"])) if i not in val_ids]
    val_ids = val_ids or tr_ids[-1:]
    take = lambda d, ids: {k: [v[i] for i in ids] for k, v in d.items()}
    return dict(train=take(tr, tr_ids), val=take(tr, val_ids), test=te)


t0 = time.time()
REAL_ROOT = find_real_root()
REAL = split_real(REAL_ROOT) if REAL_ROOT else None
if REAL:
    for k, d in REAL.items():
        lab = np.concatenate(d["labels"]); raw = np.concatenate(d["vis_raw"])
        games = sorted({c.split("/")[0] for c in d["clips"]}, key=lambda g: int(g[4:]))
        print(f"real {k:5s}: {len(d['clips']):2d} clips, {len(lab):5d} frames ({', '.join(games)}) | "
              f"ball present {np.mean(lab[:, 2] > 0):.1%} | hard {np.mean(raw == 2):.1%} | occluded {np.mean(raw == 3):.1%}")
    print(f"loaded from {REAL_ROOT} in {time.time() - t0:.0f}s")
else:
    print("Real tennis dataset: not found → real-data sections skipped (synthetic only)")

In [ ]:
def split_stats(name, d):
    lab = np.concatenate(list(d["labels"])); row = dict(split=name, clips=len(d["labels"]), frames=len(lab),
                                                        ball_present=np.mean(lab[:, 2] > 0), occluded=np.mean(lab[:, 2] == 2))
    if "vis_raw" in d:
        raw = np.concatenate(d["vis_raw"])
        row.update(games=", ".join(sorted({c.split("/")[0][4:] for c in d["clips"]}, key=int)), easy=np.mean(raw == 1), hard=np.mean(raw == 2))
    return row

rows = [split_stats(f"synthetic {k}", DATA[k]) for k in ("train", "val", "test")]
if REAL:
    rows += [split_stats(f"real {k}", REAL[k]) for k in ("train", "val", "test")]
DATASET = pd.DataFrame(rows).set_index("split")
DATASET.to_csv(RES / f"dataset_summary_{PROFILE}.csv")
(RES / f"dataset_summary_{PROFILE}.md").write_text(
    "# Datasets used\n\n" + DATASET.to_markdown(floatfmt=".3f") +
    "\n\nSynthetic: procedural broadcast renderer (§3). Real: TrackNet tennis dataset (Huang et al., 2019), "
    "10 broadcast matches at 1280×720, split by match (fine-tune games 1–8, test games 9–10).\n", encoding="utf-8")
save_table(DATASET, "tab00_datasets", f"Datasets and splits (frames resized to {W}×{H})", fmt="{:.3f}", col_width=1.05)

if REAL:
    def pick(split, n, distinct_games):
        d, out, seen = REAL[split], [], set()
        for c, name in enumerate(d["clips"]):
            key = name.split("/")[0] if distinct_games else name
            if key in seen or len(out) >= n:
                continue
            t = min(int(np.argmax(d["vis_raw"][c] == 1)) + len(d["vis_raw"][c]) // 3, len(d["frames"][c]) - 1)
            seen.add(key); out.append((split, c, t))
        return out
    picks = pick("train", 3, True) + pick("test", 3, False)      # 3 fine-tuning games / 3 test clips (games 9-10)
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    for ax, (split, c, t) in zip([*axes[0][:len(picks[:3])], *axes[1]], picks):
        d = REAL[split]; ax.imshow(d["frames"][c][t]); no_axes(ax)
        x, y, vis = d["labels"][c][t, :3]
        if vis > 0:
            ax.add_patch(Circle((x, y), 7 * SC / 1.8, fill=False, ec=PAL[2], lw=1.6))
        ax.set_title(f"{split} · {d['clips'][c]} · frame {t} · TrackNet vis {d['vis_raw'][c][t]}", fontsize=8.5, loc="left")
    for ax in axes.ravel():
        if not ax.images:
            ax.axis("off")
    fig.suptitle("Real TrackNet tennis frames (green ring = labelled ball). Top: fine-tuning games, bottom: test games",
                 x=0.01, ha="left", fontweight="bold")
    plt.tight_layout()
    savefig(fig, "fig20_real_dataset")

## 5. Models
All models share one modular implementation, so the ablation changes **exactly one thing at a time**.

* `UNetV2`: the TrackNetV2 backbone (VGG-style encoder 2-2-3-3 convs, symmetric decoder with skip concatenation, MIMO 3-heatmap head). `dw=True` switches to depthwise-separable convolutions (**M4**).
* `MDD`: Motion Direction Decoupling. `per_channel=False` is **faithful V5** (one shared α, β). `per_channel=True, direction=True` is **M2**.
* `TSATTHead`: the R-STR Transformer (patch embedding → *divided* spatial/temporal attention blocks → linear → PixelShuffle). It is zero-initialised so that it starts as an identity residual. `full` = patch 8, d = 64, 2 blocks, 4 heads; `slim` (**M3**) = patch 16, d = 32, 1 block, 2 heads.
* `TrackNet`: wires these together. `consistent=False` reproduces V5's train/inference input mismatch; `consistent=True` is **M3**.

In [ ]:
def conv_block(cin, cout, n, dw=False):
    layers = []
    for i in range(n):
        ci = cin if i == 0 else cout
        if dw:   # M4: depthwise 3x3 + pointwise 1x1
            layers += [nn.Conv2d(ci, ci, 3, padding=1, groups=ci, bias=False), nn.BatchNorm2d(ci), nn.ReLU(inplace=True),
                       nn.Conv2d(ci, cout, 1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        else:
            layers += [nn.Conv2d(ci, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
    return nn.Sequential(*layers)


class UNetV2(nn.Module):
    """TrackNetV2 backbone: 64·2 / 128·2 / 256·3 / 512·3 encoder, mirrored decoder with skips (× width)."""
    def __init__(self, cin, n_out=3, width=1.0, dw=False):
        super().__init__()
        c = [max(8, int(round(64 * width * m))) for m in (1, 2, 4, 8)]
        self.e1, self.e2 = conv_block(cin, c[0], 2), conv_block(c[0], c[1], 2, dw)
        self.e3, self.e4 = conv_block(c[1], c[2], 3, dw), conv_block(c[2], c[3], 3, dw)
        self.d3 = conv_block(c[3] + c[2], c[2], 3, dw)
        self.d2 = conv_block(c[2] + c[1], c[1], 2, dw)
        self.d1 = conv_block(c[1] + c[0], c[0], 2, dw)
        self.head = nn.Conv2d(c[0], n_out, 1)

    def forward(self, x):
        up = lambda t: F.interpolate(t, scale_factor=2, mode="nearest")
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1, 2))
        e3 = self.e3(F.max_pool2d(e2, 2))
        b = self.e4(F.max_pool2d(e3, 2))
        d = self.d3(torch.cat([up(b), e3], 1))
        d = self.d2(torch.cat([up(d), e2], 1))
        d = self.d1(torch.cat([up(d), e1], 1))
        return self.head(d)


class MDD(nn.Module):
    """Motion Direction Decoupling (TrackNetV5 Eq. 1-3).  per_channel/direction = our M2."""
    def __init__(self, per_channel=False, direction=False):
        super().__init__()
        n = 4 if per_channel else 1                       # (2 frame pairs) x (2 polarities)
        self.per_channel, self.direction = per_channel, direction
        self.alpha = nn.Parameter(torch.full((n,), 0.5))
        self.beta = nn.Parameter(torch.full((n,), 0.1))
        self.out_ch = 4 + (2 if direction else 0)

    def k_m(self):
        k = 5.0 / (0.45 * torch.tanh(self.alpha).abs() + 1e-2)
        m = 0.6 * torch.tanh(self.beta)
        shape = (1, 2, 2, 1, 1) if self.per_channel else (1, 1, 1, 1, 1)
        return k.view(shape), m.view(shape)

    def forward(self, x):
        rgb = x.view(x.shape[0], 3, 3, *x.shape[-2:])                     # [B, T, C, H, W]
        g = 0.299 * rgb[:, :, 0] + 0.587 * rgb[:, :, 1] + 0.114 * rgb[:, :, 2]
        d = torch.stack([g[:, 1] - g[:, 0], g[:, 2] - g[:, 1]], 1)          # [B, 2 pairs, H, W]
        P = torch.stack([F.relu(d), F.relu(-d)], 2)                         # [B, 2 pairs, 2 polarities, H, W]
        k, m = self.k_m()
        A = torch.sigmoid(k * (P.abs() - m))                                # Eq. 2
        return A

    def build_input(self, x, A):
        f = x.view(x.shape[0], 3, 3, *x.shape[-2:])
        parts = [f[:, 0]]
        for p in range(2):
            parts.append(A[:, p])
            if self.direction:
                parts.append((A[:, p, 0] - A[:, p, 1]).unsqueeze(1))       # explicit direction channel
            parts.append(f[:, p + 1])
        return torch.cat(parts, 1)


class Attention(nn.Module):
    """Multi-head self-attention on fused scaled_dot_product_attention (memory-efficient kernel on CPU and GPU)."""
    def __init__(self, dim, heads):
        super().__init__()
        self.h = heads
        self.qkv, self.o = nn.Linear(dim, 3 * dim), nn.Linear(dim, dim)

    def forward(self, x):
        B, N, D = x.shape
        q, k, v = self.qkv(x).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        return self.o(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, D))


class STBlock(nn.Module):
    """Divided space-time attention block (TimeSformer style), pre-norm."""
    def __init__(self, dim, heads, mlp=2):
        super().__init__()
        self.ns, self.nt, self.nm = nn.LayerNorm(dim), nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.sa, self.ta = Attention(dim, heads), Attention(dim, heads)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * mlp), nn.GELU(), nn.Linear(dim * mlp, dim))

    def forward(self, z):                                  # z: [B, T, N, D]
        B, T, N, D = z.shape
        s = z.reshape(B * T, N, D)
        s = s + self.sa(self.ns(s))
        t = s.reshape(B, T, N, D).transpose(1, 2).reshape(B * N, T, D)
        t = t + self.ta(self.nt(t))
        z = t.reshape(B, N, T, D).transpose(1, 2)
        return z + self.mlp(self.nm(z))


class TSATTHead(nn.Module):
    """R-STR residual head: (draft, motion gate) per frame -> residual logits Δ at full resolution."""
    def __init__(self, Hh, Ww, T=3, in_ch=2, patch=8, dim=64, depth=2, heads=4):
        super().__init__()
        self.p, self.h, self.w = patch, Hh // patch, Ww // patch
        self.embed = nn.Conv2d(in_ch, dim, patch, patch)
        self.pos_s = nn.Parameter(torch.randn(1, 1, self.h * self.w, dim) * 0.02)   # factorised positional enc.
        self.pos_t = nn.Parameter(torch.randn(1, T, 1, dim) * 0.02)
        self.blocks = nn.ModuleList([STBlock(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.proj = nn.Linear(dim, patch * patch)
        nn.init.zeros_(self.proj.weight); nn.init.zeros_(self.proj.bias)             # starts as identity residual
        self.shuffle = nn.PixelShuffle(patch)

    def forward(self, draft, gate):                       # [B, T, H, W] each
        B, T, Hh, Ww = draft.shape
        x = torch.stack([torch.sigmoid(draft), gate], 2).reshape(B * T, 2, Hh, Ww)
        z = self.embed(x).flatten(2).transpose(1, 2).reshape(B, T, -1, self.embed.out_channels)
        z = z + self.pos_s + self.pos_t
        for blk in self.blocks:
            z = blk(z)
        z = self.proj(self.norm(z))                       # [B, T, N, p*p]
        z = z.reshape(B * T, self.h, self.w, -1).permute(0, 3, 1, 2)
        return self.shuffle(z).reshape(B, T, Hh, Ww)


RSTR_CFG = {"full": dict(patch=8, dim=64, depth=2, heads=4), "slim": dict(patch=16, dim=32, depth=1, heads=2)}


class TrackNet(nn.Module):
    def __init__(self, mdd=None, rstr=None, consistent=False, dw=False, width=CFG["width"], **_):
        super().__init__()
        self.mdd = MDD(per_channel=(mdd == "m2"), direction=(mdd == "m2")) if mdd else None
        cin = 9 + (self.mdd.out_ch if self.mdd else 0)
        self.backbone = UNetV2(cin, 3, width, dw)
        self.consistent = consistent
        self.head = TSATTHead(H, W, **RSTR_CFG[rstr]) if rstr else None
        if rstr:
            self.gate_w = nn.Parameter(torch.tensor(1.0))       # draft ⊕ MDD fusion weight
            self.drop = nn.Dropout(0.1)

    @staticmethod
    def frame_gate(A):                                      # per-output-frame motion attention [B,3,H,W]
        a1, a2 = A[:, 0].amax(1), A[:, 1].amax(1)
        return torch.stack([a1, 0.5 * (a1 + a2), a2], 1)

    def forward(self, x, return_parts=False):
        A = self.mdd(x) if self.mdd else None
        inp = self.mdd.build_input(x, A) if self.mdd else x
        draft = self.backbone(inp)
        parts = dict(A=A, draft=draft)
        if self.head is None:
            return (draft, parts) if return_parts else draft
        gate = self.frame_gate(A) if A is not None else torch.zeros_like(draft)
        draft_mdd = draft + self.gate_w * gate
        if self.training:
            # V5 as written: train on Dropout(Draft) but infer on Draft_MDD.  M3: Draft_MDD in both phases.
            d_in = self.drop(draft_mdd if self.consistent else draft)
        else:
            d_in = draft_mdd
        delta = self.head(d_in, gate)
        out = d_in + delta
        parts.update(gate=gate, d_in=d_in, delta=delta)
        return (out, parts) if return_parts else out

### 5.1 Ablation ladder
Each rung adds exactly one modification on top of the previous one.

In [ ]:
VARIANTS = {
    "V2 (baseline)":       dict(mdd=None, rstr=None,   consistent=False, dw=False, target="disk",  decode="contour"),
    "V5 (faithful)":       dict(mdd="v5", rstr="full", consistent=False, dw=False, target="disk",  decode="contour"),
    "V5 +M1":              dict(mdd="v5", rstr="full", consistent=False, dw=False, target="gauss", decode="softargmax"),
    "V5 +M1+M2":           dict(mdd="m2", rstr="full", consistent=False, dw=False, target="gauss", decode="softargmax"),
    "V5 +M1+M2+M3":        dict(mdd="m2", rstr="slim", consistent=True,  dw=False, target="gauss", decode="softargmax"),
    "V5-Lite+ (M1–M4)":    dict(mdd="m2", rstr="slim", consistent=True,  dw=True,  target="gauss", decode="softargmax"),
}
FINAL_NAME = "V5-Lite+ (M1–M4)"                     # our final model (used from §5 on)
COLORS = dict(zip(VARIANTS, [MUTED, PAL[0], PAL[2], PAL[6], PAL[3], PAL[1]]))

@torch.no_grad()
def model_cost(model, n_runs=12):
    model.eval()
    x = torch.rand(1, 9, H, W, device=DEVICE)
    params = sum(p.numel() for p in model.parameters())
    with FlopCounterMode(display=False) as fc:
        model(x)
    flops = fc.get_total_flops()
    for _ in range(3):
        model(x)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_runs):
        model(x)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    return dict(params_M=params / 1e6, GFLOPs=flops / 1e9, FPS=3 * n_runs / dt)   # 3 frames per forward (MIMO)

cost_rows = []
for name, spec in VARIANTS.items():
    m = TrackNet(**spec).to(DEVICE)
    cost_rows.append(dict(model=name, **model_cost(m, n_runs=2 if SMOKE else 12)))
COST = pd.DataFrame(cost_rows).set_index("model")
display(COST.style.format({"params_M": "{:.3f}", "GFLOPs": "{:.2f}", "FPS": "{:.1f}"}))
save_table(COST, "tab01_model_cost", f"Model cost at {W}×{H} (one 3-frame MIMO forward pass; FPS on {DEVICE.type.upper()})",
           fmt="{:.3f}")

In [ ]:
def module_shapes(model):
    """(input shape, output shape, params) of every top-level block, captured with forward hooks."""
    info, hooks = {}, []
    blocks = [("MDD", model.mdd)] if model.mdd else []
    blocks += [(n, getattr(model.backbone, n)) for n in ("e1", "e2", "e3", "e4", "d3", "d2", "d1", "head")]
    blocks += [("R-STR", model.head)] if model.head is not None else []
    for n, mod in blocks:
        hooks.append(mod.register_forward_hook(
            lambda m, i, o, n=n: info.__setitem__(n, (tuple(i[0].shape[1:]), tuple(o.shape[1:]), sum(p.numel() for p in m.parameters())))))
    model.eval()
    with torch.no_grad():
        model(torch.rand(1, 9, H, W, device=DEVICE))
    for h in hooks:
        h.remove()
    return info

STAGE = {"MDD": "motion decoupling", "e1": "encoder 1 (conv ×2)", "e2": "encoder 2 (conv ×2)", "e3": "encoder 3 (conv ×3)",
         "e4": "bottleneck (conv ×3)", "d3": "decoder 3 (conv ×3)", "d2": "decoder 2 (conv ×2)", "d1": "decoder 1 (conv ×2)",
         "head": "1×1 conv → draft (3 maps)", "R-STR": "space-time Transformer residual"}
md_parts = []
for name in ("V5 (faithful)", FINAL_NAME):
    info = module_shapes(TrackNet(**VARIANTS[name]).to(DEVICE))
    tab = pd.DataFrame([dict(block=k, role=STAGE[k], input="×".join(map(str, v[0])), output="×".join(map(str, v[1])),
                             params=v[2]) for k, v in info.items()]).set_index("block")
    display(tab.style.set_caption(name))
    md_parts.append(f"## {name}\n\n" + tab.to_markdown() + f"\n\nTotal parameters: {sum(t[2] for t in info.values()):,}\n")
(RES / f"model_summary_{PROFILE}.md").write_text(f"# Layer-by-layer summary ({W}×{H})\n\n" + "\n".join(md_parts), encoding="utf-8")

# detailed architecture diagram of the final model (shapes are measured, not hand-written)
info = module_shapes(TrackNet(**VARIANTS[FINAL_NAME]).to(DEVICE))
fig, ax = plt.subplots(figsize=(16, 6.2)); no_axes(ax); ax.set_xlim(0, 128); ax.set_ylim(-4, 44)
def blk(x, yc, w, h, title, sub, col):
    ax.add_patch(FancyBboxPatch((x, yc - h / 2), w, h, boxstyle="round,pad=0.3,rounding_size=0.8", fc=col + "26", ec=col, lw=1.6))
    ax.text(x + w / 2, yc + 0.9, title, ha="center", va="bottom", fontsize=8.2, fontweight="bold")
    ax.text(x + w / 2, yc + 0.4, sub, ha="center", va="top", fontsize=7.2, color=INK2)
def arrow(x0, y0, x1, y1, **kw):
    ax.annotate("", (x1, y1), (x0, y0), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1.1, **kw))
shape = lambda k: "×".join(map(str, info[k][1]))
pk = lambda k: f"{info[k][2] / 1e3:.1f}k params"
O, M = PAL[1], MUTED
blk(0.5, 30, 9, 9, "Input", f"3 RGB frames\n9×{H}×{W}", M)
blk(11.5, 30, 11, 11, "MDD-M2", f"P± per pair/polarity\n+ direction A⁺−A⁻\n{shape('MDD')}\n{info['MDD'][2]} params", O)
arrow(9.5, 30, 11.5, 30)
arrow(22.5, 31.5, 25, 33); ax.text(17, 22.5, f"concat with RGB\n→ {info['e1'][0][0]} channels", ha="center", va="top", fontsize=7.5, color=INK2)
xs_e, ys = [25, 34.5, 44, 53.5], [33, 25, 17, 9]
xs_d = [63, 72.5, 82]
for (n, x, y) in zip(("e1", "e2", "e3", "e4"), xs_e, ys):
    blk(x, y, 8.5, 6.5, n + (" (3×3 conv)" if n == "e1" else " (DW-sep)"), f"{shape(n)}\n{pk(n)}", M if n == "e1" else O)
for (n, x, y) in zip(("d3", "d2", "d1"), xs_d, ys[2::-1]):
    blk(x, y, 8.5, 6.5, n + " (DW-sep)", f"{shape(n)}\n{pk(n)}", O)
for a, b in zip(xs_e[:-1], xs_e[1:]):
    arrow(a + 8.5, ys[xs_e.index(a)], b, ys[xs_e.index(b)] + 1.5); ax.text((a + b) / 2 + 4, ys[xs_e.index(a)] - 4.3, "max-pool ↓2", fontsize=6.5, color=INK2, ha="center")
arrow(53.5 + 8.5, 9, 63, 17 - 1.5); ax.text(64, 11, "upsample ↑2", fontsize=6.5, color=INK2)
arrow(63 + 8.5, 17, 72.5, 25 - 1.5); arrow(72.5 + 8.5, 25, 82, 33 - 1.5)
for (xe, xd, y) in ((25, 82, 33), (34.5, 72.5, 25), (44, 63, 17)):      # skip connections
    ax.annotate("", (xd, y + 2.2), (xe + 8.5, y + 2.2), arrowprops=dict(arrowstyle="-|>", color=PAL[0], lw=1, ls="--",
                connectionstyle="arc3,rad=-0.12"))
ax.text(54, 41.5, "U-Net skip connections (concatenate)", color=PAL[0], fontsize=7.5, ha="center")
blk(93, 33, 8.5, 6.5, "1×1 conv", f"draft D\n{shape('head')}", M); arrow(90.5, 33, 93, 33)
blk(91.5, 20, 11.5, 9, "D ⊕ MDD gate", "D_MDD = D + w·A\nsame input in\ntrain & test (M3)", O); arrow(97.2, 29.7, 97.2, 24.5)
blk(104.5, 20, 11.5, 10, "slim R-STR (M3)", f"patch 16, d=32\nspace + time attn\n→ Δ {shape('R-STR')}\n{pk('R-STR')}", O); arrow(103, 20, 104.5, 20)
blk(117.5, 20, 10, 9, "Output", "σ(D_MDD + Δ)\n3 heatmaps\nM1 decode → M5", O); arrow(116, 20, 117.5, 20)
ax.text(0.5, -2.5, f"TrackNetV5-Lite+ at {W}×{H}, width {CFG['width']} — total {sum(v[2] for v in info.values()):,} parameters. "
        "Orange = changed vs. TrackNetV5; shapes are C×H×W measured from the model with forward hooks.", fontsize=8, color=INK2)
ax.set_title("Proposed architecture: TrackNetV5-Lite+ (detailed)", loc="left")
savefig(fig, "fig19_architecture_detailed")

### 5.2 Algorithmic steps
**Algorithm 1: Training (one variant).**
1. **Data.**
   - Synthetic: render clips (court, ballistic ball with bounces/hits, motion blur, occluding players, camera shake, noise) with exact labels $(x, y, \text{vis})$.
   - Real: load TrackNet clips and resize to $W\times H$.
2. **Sample** a window of 3 consecutive frames $(I_{t-1}, I_t, I_{t+1})$ and apply a random horizontal flip and brightness/contrast jitter (labels flipped with it).
3. **Targets**, per frame: a binary disk of radius $30\cdot W/1280$ (V2/V5) or a Gaussian with $\sigma=\max(1, 0.8\,r_{ball})$ (**M1**). The map is all zeros when no ball is present.
4. **Motion Decoupling.**
   - Grey-level differences $\Delta_1=I_t-I_{t-1}$, $\Delta_2=I_{t+1}-I_t$; polarities $P^{\pm}=\mathrm{ReLU}(\pm\Delta)$.
   - Attention $A=\sigma(k(\alpha)(|P|-m(\beta)))$ with one shared $(\alpha,\beta)$ (V5), or one per frame pair and polarity plus a direction channel $A^+-A^-$ (**M2**).
   - Concatenate with RGB to 13 channels (V5) or 15 channels (M2).
5. **Backbone**: the U-Net encoder (3 max-pools) and decoder with skip concatenation, using 3×3 convs or depthwise-separable convs (**M4**). It outputs 3 draft logit maps $D$ (MIMO).
6. **R-STR refinement.**
   - $D_{MDD}=D+w\cdot A_t$.
   - The head input is $\mathrm{Dropout}(D)$ in training (V5) or $\mathrm{Dropout}(D_{MDD})$ in both phases (**M3**).
   - TSATTHead: patch embedding → divided spatial and temporal attention → linear → PixelShuffle, giving the residual $\Delta$. Output $=D_{in}+\Delta$.
7. **Optimise** the weighted BCE $-[(1-p)^2y\log p+p^2(1-y)\log(1-p)]$ with AdamW, gradient clipping at 5, and a ×0.1 learning-rate decay at 2/3 and 5/6 of the steps. Keep the checkpoint with the best validation F1.
8. **Real fine-tuning**: initialise from the synthetic checkpoint and repeat steps 2–7 on games 1–8 with a lower learning rate.

**Algorithm 2: Inference.** Split each clip into non-overlapping 3-frame windows → forward pass → $\sigma$ → 3 heatmaps. Decode each frame:
- V2/V5: threshold at 0.5 and take the bounding-box centre of the largest blob.
- **M1**: take the global peak $p^*$; if $p^*\ge0.5$, return the centroid of the weights $\max(p-0.1p^*,0)$ in a $(2k+1)^2$ window.

**Algorithm 3: M5 post-processing (per clip).**
1. Constant-acceleration Kalman filter on $(x,y,\dot x,\dot y,\ddot x,\ddot y)$, with measurement noise scaled by $1/\text{confidence}$.
2. Reject a detection if its Mahalanobis innovation exceeds $\chi^2_{2,0.999}=13.8$. If two consecutive rejected detections agree with each other, re-initialise the track (a hit or bounce).
3. Fill gaps of ≤ 4 frames between accepted detections with a per-axis quadratic fit in time over ≤ 3 detections on each side. Discard fills that leave the frame.

**Algorithm 4: Evaluation.** For each frame, classify the prediction as TP, FP1, FP2, FN or TN at tolerance τ. Then compute precision $=TP/(TP+FP1+FP2)$, recall $=TP/(TP+FN)$, F1, accuracy, and the mean localisation error over hits.

## 6. Training
* **Loss**: the V2/V5 weighted BCE, implemented in a numerically stable form from logits.
* **Optimiser**: AdamW with V5's **multi-step decay** (×0.1 at 2/3 and 5/6 of training, the same proportions as the paper's epochs 20/25 of 30). The paper uses lr = 1e‑4 with batch 2 for 30 epochs on ~20k frames. Our budget is far smaller (`CFG['steps']`, see §2), so we raise the learning rate to 1e‑3 (GPU) / 2e‑3 (CPU). A pilot run showed that disk-target models need a few hundred updates before any pixel crosses the 0.5 threshold, so the schedule favours many small steps. The recipe is identical for every variant.
* **Model selection**: the checkpoint with the best validation F1 is kept, evaluated every `eval_every` steps.
* Identical seeds, data order and step budget for every variant, so differences come from the architecture and targets only.
* Checkpoints are cached in `checkpoints/` (git-ignored). Re-running the notebook re-uses them unless `TN_RETRAIN=1`.

In [ ]:
def wbce_loss(logits, y):
    """TrackNetV2/V5 weighted BCE: -[(1-p)^2 y log p + p^2 (1-y) log(1-p)], stable from logits."""
    p = torch.sigmoid(logits)
    return -(((1 - p) ** 2) * y * F.logsigmoid(logits) + (p ** 2) * (1 - y) * F.logsigmoid(-logits)).mean()


def decode_contour(hm, thr):
    """TrackNetV2/V3/V5 decoding: threshold -> largest connected blob -> bbox centre."""
    mask = (hm > thr).astype(np.uint8)
    if not mask.any():
        return None, 0.0
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    x, y, w, h = max((cv2.boundingRect(c) for c in cnts), key=lambda b: b[2] * b[3])
    return (x + (w - 1) / 2, y + (h - 1) / 2), float(hm[y:y + h, x:x + w].max())


def decode_softargmax(hm, thr, k=None, rel=0.1):
    """M1 decoding: global peak -> centroid of the peak's weights max(p - rel*peak, 0) in a (2k+1)^2 window.

    Thresholding by *value* keeps the weights symmetric about the true centre, so the estimate is unbiased
    at sub-pixel level (a plain centroid in a window centred on the integer peak pixel is pulled towards it).
    """
    k = int(math.ceil(7 * SC)) if k is None else k              # covers the 0.1-level of the largest balls
    iy, ix = np.unravel_index(np.argmax(hm), hm.shape)
    conf = float(hm[iy, ix])
    if conf < thr:
        return None, conf
    y0, y1, x0, x1 = max(0, iy - k), min(H, iy + k + 1), max(0, ix - k), min(W, ix + k + 1)
    win = np.clip(hm[y0:y1, x0:x1] - rel * conf, 0, None)
    return (float((win * XX[y0:y1, x0:x1]).sum() / win.sum()), float((win * YY[y0:y1, x0:x1]).sum() / win.sum())), conf

DECODERS = {"contour": decode_contour, "softargmax": decode_softargmax}


@torch.no_grad()
def predict_heatmaps(model, split, batch=8):
    """Non-overlapping 3-frame MIMO windows over every clip -> list of [L,H,W] uint8 probability maps."""
    model.eval()
    frames = split["frames"]
    out = []
    for c in range(len(frames)):
        fr = frames[c]; Lc = len(fr)
        starts = list(range(0, Lc - 2, 3))
        if starts[-1] + 3 < Lc:
            starts.append(Lc - 3)
        hm = np.zeros((Lc, H, W), np.uint8)
        for b in range(0, len(starts), batch):
            ss = starts[b:b + batch]
            x = np.stack([fr[s:s + 3] for s in ss]).astype(np.float32) / 255.0      # [B,3,H,W,3]
            x = torch.from_numpy(x.transpose(0, 1, 4, 2, 3).reshape(len(ss), 9, H, W)).to(DEVICE)
            p = torch.sigmoid(model(x)).cpu().numpy()
            for j, s in enumerate(ss):
                hm[s:s + 3] = np.round(p[j] * 255).astype(np.uint8)
        out.append(hm)
    return out


def decode_all(heatmaps, decode, thr):
    preds, confs = [], []
    for hm in heatmaps:
        P = np.full((len(hm), 2), np.nan, np.float32); C = np.zeros(len(hm), np.float32)
        for t in range(len(hm)):
            xy, cf = DECODERS[decode](hm[t].astype(np.float32) / 255.0, thr)
            if xy is not None:
                P[t] = xy
            C[t] = cf
        preds.append(P); confs.append(C)
    return preds, confs


def evaluate(preds, labels, tol=CFG["tol"]):
    P = np.concatenate(preds); Lb = np.concatenate([np.asarray(l) for l in labels])
    present, has = Lb[:, 2] > 0, ~np.isnan(P[:, 0])
    dist = np.where(present & has, np.hypot(P[:, 0] - Lb[:, 0], P[:, 1] - Lb[:, 1]), np.nan)
    TP = int(np.sum(present & has & (dist <= tol))); FP1 = int(np.sum(present & has & (dist > tol)))
    FP2 = int(np.sum(~present & has)); FN = int(np.sum(present & ~has)); TN = int(np.sum(~present & ~has))
    prec = TP / max(TP + FP1 + FP2, 1); rec = TP / max(TP + FN, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    hit_err = dist[present & has & (dist <= tol)]
    return dict(acc=(TP + TN) / len(P), prec=prec, rec=rec, f1=f1, TP=TP, FP1=FP1, FP2=FP2, FN=FN, TN=TN,
                err_mean=float(np.nanmean(hit_err)) if hit_err.size else np.nan,
                err_median=float(np.nanmedian(hit_err)) if hit_err.size else np.nan)

In [ ]:
def variant_tag(name):
    """File-system-safe name for a variant's checkpoint / history files."""
    return name.replace(" ", "_").replace("+", "p").replace("(", "").replace(")", "").replace("–", "-")


def validate(model, spec, val):
    hm = predict_heatmaps(model, val)
    pr, _ = decode_all(hm, spec["decode"], CFG["det_thr"])
    return evaluate(pr, val["labels"])


def train_variant(name, spec, train=None, val=None, steps=None, lr=None, eval_every=None, init=None, suffix="",
                  log_every=25):
    """Train one variant (defaults: synthetic data, CFG schedule). init= starts from another model (fine-tuning)."""
    train = DATA["train"] if train is None else train
    val = DATA["val"] if val is None else val
    S, lr, eval_every = steps or CFG["steps"], lr or CFG["lr"], eval_every or CFG["eval_every"]
    tag = variant_tag(name) + suffix
    ck, hist_f = CKPT / f"{tag}_{PROFILE}.pt", RES / f"history_{tag}_{PROFILE}.json"
    seed_everything()
    model = TrackNet(**spec).to(DEVICE)
    if ck.exists() and hist_f.exists() and not RETRAIN and not SMOKE:
        model.load_state_dict(torch.load(ck, map_location=DEVICE))
        print(f"[{name}{suffix}] loaded cached checkpoint {ck.name}")
        return model, json.loads(hist_f.read_text())
    if init is not None:
        model.load_state_dict(init.state_dict())
    loader = DataLoader(WindowDataset(train, spec["target"], augment=True), batch_size=CFG["batch"],
                        shuffle=True, drop_last=True, num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0,
                        pin_memory=DEVICE.type == "cuda", generator=torch.Generator().manual_seed(SEED))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, [int(S * 2 / 3), int(S * 5 / 6)], 0.1)
    hist = dict(step=[], loss=[], val_step=[], val_f1=[], val_err=[], time_s=0.0)
    best, best_state, step, run, t0 = -1, None, 0, [], time.time()
    if init is not None:                       # the starting point is a candidate too: fine-tuning can't make it worse
        ev = validate(model, spec, val)
        hist["val_step"].append(0); hist["val_f1"].append(ev["f1"]); hist["val_err"].append(ev["err_mean"])
        best, best_state = ev["f1"], copy.deepcopy(model.state_dict())
        print(f"[{name}{suffix}] step    0/{S} | val F1 {ev['f1']:.3f} (starting point)")
    while step < S:
        for x, y, _ in loader:
            model.train()
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = wbce_loss(model(x), y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sched.step(); step += 1
            run.append(loss.item())
            if step % log_every == 0 or step == S:
                hist["step"].append(step); hist["loss"].append(float(np.mean(run))); run = []
            if step % eval_every == 0 or step == S:
                ev = validate(model, spec, val)
                hist["val_step"].append(step); hist["val_f1"].append(ev["f1"]); hist["val_err"].append(ev["err_mean"])
                print(f"[{name}{suffix}] step {step:4d}/{S} | loss {hist['loss'][-1] if hist['loss'] else loss.item():.5f} | val F1 {ev['f1']:.3f} "
                      f"P {ev['prec']:.3f} R {ev['rec']:.3f} | err {ev['err_mean']:.2f}px | {time.time()-t0:.0f}s")
                if ev["f1"] > best:
                    best, best_state = ev["f1"], copy.deepcopy(model.state_dict())
            if step >= S:
                break
    hist["time_s"] = time.time() - t0
    model.load_state_dict(best_state)
    if not SMOKE:
        torch.save(best_state, ck); hist_f.write_text(json.dumps(hist))
    return model, hist

In [ ]:
MODELS, HIST = {}, {}
for name, spec in VARIANTS.items():
    MODELS[name], HIST[name] = train_variant(name, spec)
    print(f"[{name}] done in {HIST[name]['time_s']/60:.1f} min\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for name in VARIANTS:
    h = HIST[name]
    axes[0].plot(h["step"], h["loss"], color=COLORS[name], label=name, lw=1.6)
    axes[1].plot(h["val_step"], h["val_f1"], "o-", color=COLORS[name], label=name, ms=4, lw=1.6)
axes[0].set(title="Training loss (weighted BCE)", xlabel="step", yscale="log")
axes[1].set(title=f"Validation F1 (τ = {CFG['tol']:.1f}px)", xlabel="step", ylim=(0, 1))
axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout()
savefig(fig, "fig06_training_curves")
display(pd.DataFrame({n: dict(train_min=HIST[n]["time_s"] / 60, best_val_f1=max(HIST[n]["val_f1"])) for n in VARIANTS}).T.round(3))

## 7. Evaluation & ablation study
All models are evaluated on held-out **test clips** (unseen courts, players, lighting and rallies) with the TrackNet protocol. Each model uses its own decoder: contour/bbox for disk-trained models (as in V2/V5), soft-argmax for M1+.

In [ ]:
TEST_HM, TEST_PRED, TEST_CONF = {}, {}, {}
rows = []
for name, spec in VARIANTS.items():
    TEST_HM[name] = predict_heatmaps(MODELS[name], DATA["test"])
    TEST_PRED[name], TEST_CONF[name] = decode_all(TEST_HM[name], spec["decode"], CFG["det_thr"])
    rows.append(dict(model=name, **evaluate(TEST_PRED[name], DATA["test"]["labels"])))
RESULTS = pd.DataFrame(rows).set_index("model").join(COST)
RESULTS.to_csv(RES / f"ablation_{PROFILE}.csv")
save_table(RESULTS[["f1", "prec", "rec", "acc", "err_mean", "TP", "FP1", "FP2", "FN", "params_M", "GFLOPs", "FPS"]],
           "tab02_synthetic_ablation", f"Ablation on the synthetic test set (τ = {CFG['tol']:.1f} px at {W}×{H})", col_width=0.85)
fmt = {"acc": "{:.3f}", "prec": "{:.3f}", "rec": "{:.3f}", "f1": "{:.3f}", "err_mean": "{:.2f}", "err_median": "{:.2f}",
       "params_M": "{:.3f}", "GFLOPs": "{:.2f}", "FPS": "{:.1f}"}
display(RESULTS.style.format(fmt).background_gradient(subset=["f1"], cmap="Blues").background_gradient(subset=["err_mean"], cmap="Blues_r"))

In [ ]:
def delta_table(res):
    names = list(res.index)
    rows = []
    for a, b in zip(names[:-1], names[1:]):
        ra, rb = res.loc[a], res.loc[b]
        rows.append({"step": f"{a} → {b}", "ΔF1": rb.f1 - ra.f1, "ΔPrecision": rb.prec - ra.prec, "ΔRecall": rb.rec - ra.rec,
                     "Δerr (px)": rb.err_mean - ra.err_mean, "Δparams (M)": rb.params_M - ra.params_M,
                     "ΔGFLOPs": rb.GFLOPs - ra.GFLOPs, "ΔFPS": rb.FPS - ra.FPS})
    return pd.DataFrame(rows).set_index("step")

DELTAS = delta_table(RESULTS)
display(DELTAS.style.format("{:+.3f}").map(lambda v: f"color: {PAL[5]}" if v > 0 else (f"color: {PAL[7]}" if v < 0 else ""),
                                            subset=["ΔF1", "ΔPrecision", "ΔRecall"]))

In [ ]:
names = list(VARIANTS)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
xs = np.arange(len(names))
short = [n.replace(" (baseline)", "\n(baseline)").replace(" (faithful)", "\n(faithful)").replace(" (M1–M4)", "\n(M1–M4)").replace("V5 +", "V5\n+") for n in names]
for ax, col, ttl in [(axes[0], "f1", f"F1 @ τ={CFG['tol']:.1f}px (higher is better)"),
                     (axes[1], "err_mean", "Mean localisation error of hits, px (lower is better)")]:
    vals = RESULTS[col].values
    ax.bar(xs, vals, color=[COLORS[n] for n in names], width=0.62)
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.3f}" if col == "f1" else f"{v:.2f}", ha="center", va="bottom", fontsize=8, color=INK2)
    ax.set_xticks(xs, short, fontsize=7.5); ax.set_title(ttl, loc="left")
    if col == "f1":
        ax.set_ylim(max(0, vals.min() - 0.15), min(1.0, vals.max() + 0.06))
stack = RESULTS[["TP", "FP1", "FP2", "FN"]]
bottom = np.zeros(len(names))
for c, colr in zip(stack.columns, [PAL[0], PAL[1], PAL[3], PAL[7]]):
    axes[2].bar(xs, stack[c].values, bottom=bottom, color=colr, label=c, width=0.62, edgecolor="#fcfcfb", linewidth=1)
    bottom += stack[c].values
axes[2].set_xticks(xs, short, fontsize=7.5); axes[2].set_title("Outcome counts on test frames", loc="left")
axes[2].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22), fontsize=8)
axes[2].set_ylim(0, stack.sum(axis=1).max() * 1.08)
plt.tight_layout()
savefig(fig, "fig07_ablation_bars")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, col, lbl in [(axes[0], "GFLOPs", "GFLOPs per 3-frame forward pass"), (axes[1], "FPS", "CPU throughput, frames / s (batch 1)")]:
    for n in names:
        ax.scatter(RESULTS.loc[n, col], RESULTS.loc[n, "f1"], s=110, color=COLORS[n], edgecolor="#fcfcfb", linewidth=2, zorder=3, label=n)
        ax.annotate(n, (RESULTS.loc[n, col], RESULTS.loc[n, "f1"]), xytext=(6, 4), textcoords="offset points", fontsize=7.5, color=INK2)
    ax.set(xlabel=lbl, ylabel="test F1", title=f"Accuracy vs. {'compute' if col == 'GFLOPs' else 'speed'}")
plt.tight_layout()
savefig(fig, "fig08_accuracy_vs_cost")

### 7.1 Strictness of localisation: F1 vs. tolerance τ and the error distribution
This is where **M1** should show its effect most. Disk-trained models place their output somewhere inside a ~13-px blob, so they only score well when τ is loose. Gaussian + soft-argmax models should keep their F1 at strict τ.

In [ ]:
taus = np.linspace(0.5, 8, 16) * SC
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for n in names:
    f1s = [evaluate(TEST_PRED[n], DATA["test"]["labels"], tol=t)["f1"] for t in taus]
    axes[0].plot(taus, f1s, color=COLORS[n], label=n)
    P = np.concatenate(TEST_PRED[n]); Lb = np.concatenate(DATA["test"]["labels"])
    ok = (Lb[:, 2] > 0) & ~np.isnan(P[:, 0])
    d = np.sort(np.hypot(P[ok, 0] - Lb[ok, 0], P[ok, 1] - Lb[ok, 1]))
    axes[1].plot(d, np.arange(1, len(d) + 1) / len(d), color=COLORS[n], label=n)
axes[0].axvline(CFG["tol"], color=MUTED, ls="--", lw=1); axes[0].text(CFG["tol"] + 0.1, 0.03, "headline τ", color=INK2, fontsize=8)
axes[0].set(title="F1 as the tolerance τ tightens", xlabel="τ (px at working resolution)", ylabel="F1", ylim=(0, 1))
axes[1].set(title="CDF of localisation error (detected frames)", xlabel="error (px)", ylabel="fraction ≤ error", xlim=(0, 10 * SC))
axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8)
plt.tight_layout()
savefig(fig, "fig09_tolerance_and_error_cdf")
strict = pd.DataFrame({n: {f"F1@τ={t}": evaluate(TEST_PRED[n], DATA["test"]["labels"], tol=t)["f1"] for t in np.round(np.array([1.0, 2.0, 3.0, 5.0]) * SC, 1)} for n in names}).T
display(strict.style.format("{:.3f}").background_gradient(cmap="Blues", axis=0))

In [ ]:
thrs = np.linspace(0.1, 0.9, 9)
fig, ax = plt.subplots(figsize=(6.2, 4))
for n in names:
    pts = []
    for th in thrs:
        pr, _ = decode_all(TEST_HM[n], VARIANTS[n]["decode"], th)
        e = evaluate(pr, DATA["test"]["labels"])
        pts.append((e["rec"], e["prec"]))
    pts = np.array(pts)
    ax.plot(pts[:, 0], pts[:, 1], "o-", color=COLORS[n], label=n, ms=3.5, lw=1.6)
ax.set(title="Precision–recall over detection thresholds 0.1 … 0.9", xlabel="recall", ylabel="precision")
ax.legend(fontsize=7.5, loc="lower left")
savefig(fig, "fig10_precision_recall")

### 7.2 Robustness: where do the models fail?
Hit-rate (TP / ball-present frames) broken down by **speed**, **ball size**, **proximity to a player** (clutter) and **occlusion**. To keep the charts readable we compare the paper's two models with our final model.

In [ ]:
def hit_rate_by(pred_list, labels, mask_fn, tol=CFG["tol"]):
    P = np.concatenate(pred_list); Lb = np.concatenate(labels)
    m = mask_fn(Lb) & (Lb[:, 2] > 0)
    d = np.hypot(P[m, 0] - Lb[m, 0], P[m, 1] - Lb[m, 1])
    return float(np.nansum(d <= tol) / max(m.sum(), 1)), int(m.sum())

Lb_test = DATA["test"]["labels"]
groups = {
    "speed": [(f"<{4*SC:.0f} px/f", lambda L: L[:, 3] < 4 * SC), (f"{4*SC:.0f}–{8*SC:.0f}", lambda L: (L[:, 3] >= 4 * SC) & (L[:, 3] < 8 * SC)),
              (f"{8*SC:.0f}–{12*SC:.0f}", lambda L: (L[:, 3] >= 8 * SC) & (L[:, 3] < 12 * SC)), (f"≥{12*SC:.0f}", lambda L: L[:, 3] >= 12 * SC)],
    "ball radius": [(f"<{1.6*SC:.1f} px", lambda L: L[:, 4] < 1.6 * SC), (f"{1.6*SC:.1f}–{2.2*SC:.1f}", lambda L: (L[:, 4] >= 1.6 * SC) & (L[:, 4] < 2.2 * SC)),
                    (f"≥{2.2*SC:.1f}", lambda L: L[:, 4] >= 2.2 * SC)],
    "context": [("open court", lambda L: (L[:, 5] == 0) & (L[:, 2] == 1)), ("near player", lambda L: (L[:, 5] == 1) & (L[:, 2] == 1)),
                ("occluded", lambda L: L[:, 2] == 2)],
}
KEY = ["V2 (baseline)", "V5 (faithful)", "V5-Lite+ (M1–M4)"]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), gridspec_kw=dict(width_ratios=[4, 3, 3]))
robust_rows = []
for ax, (gname, bins) in zip(axes, groups.items()):
    w = 0.26
    for k, n in enumerate(KEY):
        vals = [hit_rate_by(TEST_PRED[n], Lb_test, fn) for _, fn in bins]
        ax.bar(np.arange(len(bins)) + (k - 1) * w, [v[0] for v in vals], width=w - 0.02, color=[MUTED, PAL[0], PAL[1]][k], label=n)
        for (b, _), (v, cnt) in zip(bins, vals):
            robust_rows.append(dict(group=gname, bin=b, model=n, hit_rate=v, frames=cnt))
    counts = [hit_rate_by(TEST_PRED[KEY[0]], Lb_test, fn)[1] for _, fn in bins]
    ax.set_xticks(np.arange(len(bins)), [f"{b}\n(n={c})" for (b, _), c in zip(bins, counts)], fontsize=8)
    ax.set(title=f"Hit-rate by {gname}", ylim=(0, 1.05))
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=3, fontsize=8.5, bbox_to_anchor=(0.5, 1.06))
plt.tight_layout()
savefig(fig, "fig11_robustness")
ROBUST = pd.DataFrame(robust_rows).pivot_table(index=["group", "bin"], columns="model", values="hit_rate")[KEY]
display(ROBUST.style.format("{:.3f}"))

## 8. M5: physics-aware temporal post-processing
The network is a 3-frame detector. It cannot reason about a ball that stays hidden for several frames, and with no memory it cannot reject a single implausible detection (a crowd speck, a white shoe) either. V5's authors name **false positives in complex backgrounds** as their main remaining limitation. M5 adds a cheap, training-free tracker on top:

1. **Constant-acceleration Kalman filter**. State $(x,y,\dot x,\dot y,\ddot x,\ddot y)$, which is exactly the ballistic motion model between bounces and hits. Measurement noise shrinks with detection confidence.
2. **χ² gating**. A detection whose Mahalanobis innovation exceeds $\chi^2_{2,0.999}=13.8$ is rejected as an outlier (removing FP1/FP2). If ≥ 2 consecutive rejected detections agree with each other, the track is **re-initialised**. That is the signature of a real hit or bounce, not of noise.
3. **Ballistic gap filling**. Gaps of ≤ 4 frames between accepted detections are filled by fitting a 2nd-order polynomial in time (gravity ⇒ parabola) to up to 3 detections on each side. This recovers occluded frames.

Cost: microseconds per frame, no parameters, no training. Risk: a hit *inside* a gap is interpolated wrongly, and over-aggressive gating can drop true detections after an abrupt direction change.

In [ ]:
def kalman_refine(P, C, gate=13.8, max_gap=4, q=0.6 * SC ** 2, r0=1.0 * SC, fill=True):
    Lc = len(P)
    F_ = np.eye(6); F_[0, 2] = F_[1, 3] = 1; F_[2, 4] = F_[3, 5] = 1; F_[0, 4] = F_[1, 5] = 0.5
    H_ = np.zeros((2, 6)); H_[0, 0] = H_[1, 1] = 1
    Q = np.diag([0.25, 0.25, 0.5, 0.5, 1.0, 1.0]) * q
    acc = np.full((Lc, 2), np.nan, np.float32)
    x = Pm = None; misses = 0; pending = []

    def init(z):
        return np.array([z[0], z[1], 0, 0, 0, 0], float), np.diag([4.0, 4.0, 100.0, 100.0, 4.0, 4.0]) * SC ** 2

    for t in range(Lc):
        if x is not None:
            x, Pm = F_ @ x, F_ @ Pm @ F_.T + Q
        z = P[t]
        if np.isnan(z[0]):
            misses += 1
            if misses > max_gap:
                x = None; pending = []
            continue
        R = np.eye(2) * (r0 / max(C[t], 0.2)) ** 2
        if x is None:
            x, Pm = init(z); acc[t] = z; misses = 0; pending = []
            continue
        y_ = z - H_ @ x; S = H_ @ Pm @ H_.T + R
        d2 = float(y_ @ np.linalg.solve(S, y_))
        if d2 <= gate:
            K = Pm @ H_.T @ np.linalg.inv(S)
            x, Pm = x + K @ y_, (np.eye(6) - K @ H_) @ Pm
            acc[t] = z; misses = 0; pending = []
        else:
            pending.append(t)
            if len(pending) >= 2 and np.hypot(*(P[pending[-1]] - P[pending[-2]])) < 25 * SC and pending[-1] - pending[-2] <= 2:
                x, Pm = init(P[pending[-1]])                 # consistent new evidence -> re-initialise (hit / bounce)
                x[2:4] = (P[pending[-1]] - P[pending[-2]]) / (pending[-1] - pending[-2])
                for tt in pending[-2:]:
                    acc[tt] = P[tt]
                pending = []; misses = 0
    if not fill:
        return acc
    out = acc.copy()
    ok = np.where(~np.isnan(acc[:, 0]))[0]
    for a, b in zip(ok[:-1], ok[1:]):
        gap = b - a - 1
        if 1 <= gap <= max_gap:
            left = ok[(ok <= a)][-3:]; right = ok[(ok >= b)][:3]
            ts = np.concatenate([left, right])
            if len(ts) < 3:
                continue
            for dim in (0, 1):
                coef = np.polyfit(ts, acc[ts, dim], 2 if len(ts) >= 4 else 1)
                out[a + 1:b, dim] = np.polyval(coef, np.arange(a + 1, b))
            if np.any((out[a + 1:b, 0] < 0) | (out[a + 1:b, 0] >= W) | (out[a + 1:b, 1] < 0) | (out[a + 1:b, 1] >= H)):
                out[a + 1:b] = np.nan                        # never fill a path that leaves the frame
    return out

In [ ]:
m5_rows = []
for n in names:
    for mode, kw in [("raw", None), ("+ gating only", dict(fill=False)), ("+ M5 (gating + gap filling)", dict(fill=True))]:
        pr = TEST_PRED[n] if kw is None else [kalman_refine(p, c, **kw) for p, c in zip(TEST_PRED[n], TEST_CONF[n])]
        m5_rows.append(dict(model=n, post=mode, **evaluate(pr, Lb_test)))
M5 = pd.DataFrame(m5_rows).set_index(["model", "post"])
display(M5[["f1", "prec", "rec", "acc", "FP1", "FP2", "FN", "err_mean"]].style.format(
    {"f1": "{:.3f}", "prec": "{:.3f}", "rec": "{:.3f}", "acc": "{:.3f}", "err_mean": "{:.2f}"}))
FINAL_NAME = "V5-Lite+ (M1–M4)"
FINAL_PRED = [kalman_refine(p, c) for p, c in zip(TEST_PRED[FINAL_NAME], TEST_CONF[FINAL_NAME])]
occl_before = hit_rate_by(TEST_PRED[FINAL_NAME], Lb_test, lambda L: L[:, 2] == 2)[0]
occl_after = hit_rate_by(FINAL_PRED, Lb_test, lambda L: L[:, 2] == 2)[0]
print(f"Occluded-frame hit-rate for {FINAL_NAME}: {occl_before:.3f} → {occl_after:.3f} with M5")

In [ ]:
# trajectory before / after M5 on the test clip where M5 changes the most frames
diffs = [np.sum(np.isnan(a[:, 0]) != np.isnan(b[:, 0])) for a, b in zip(TEST_PRED[FINAL_NAME], FINAL_PRED)]
c = int(np.argmax(diffs))
Lc_, raw, ref = Lb_test[c], TEST_PRED[FINAL_NAME][c], FINAL_PRED[c]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw=dict(width_ratios=[1.3, 1]))
ax = axes[0]
ax.imshow(DATA["test"]["frames"][c][len(Lc_) // 2], alpha=0.55)
m = Lc_[:, 2] > 0
ax.plot(Lc_[m, 0], Lc_[m, 1], "-", color=PAL[2], lw=1.2, label="ground truth")
ax.scatter(raw[:, 0], raw[:, 1], s=16, color=PAL[7], label="raw detections", zorder=3)
ax.scatter(ref[:, 0], ref[:, 1], s=36, facecolor="none", edgecolor=PAL[0], lw=1.3, label="after M5", zorder=4)
no_axes(ax); ax.legend(fontsize=8, loc="lower left"); ax.set_title(f"Test clip {c}: detections vs. M5 track", loc="left")
ax = axes[1]
tt = np.arange(len(Lc_))
ax.plot(tt[m], Lc_[m, 1], color=PAL[2], lw=1.2, label="GT y(t)")
ax.scatter(tt, raw[:, 1], s=14, color=PAL[7], label="raw")
ax.scatter(tt, ref[:, 1], s=34, facecolor="none", edgecolor=PAL[0], lw=1.3, label="M5")
for t in np.where(Lc_[:, 2] == 2)[0]:
    ax.axvspan(t - 0.5, t + 0.5, color=PAL[1], alpha=0.15, lw=0)
ax.set(title="y-coordinate over time (orange bands = occluded)", xlabel="frame", ylabel="y (px)"); ax.invert_yaxis()
ax.legend(fontsize=8)
plt.tight_layout()
savefig(fig, "fig12_kalman_m5")

## 9. Visual analysis
### 9.1 What the networks output
One fast-ball test window. Rows: models. Columns: input crop, predicted heatmap, and a zoomed overlay (green ring = ground truth, ✕ = decoded prediction).

In [ ]:
Lcat = np.concatenate(Lb_test)
cand = np.where((Lcat[:, 2] == 1) & (Lcat[:, 3] > np.percentile(Lcat[Lcat[:, 2] == 1, 3], 75)))[0]
gi = int(cand[len(cand) // 2]) if len(cand) else int(np.where(Lcat[:, 2] == 1)[0][0])
c, t = divmod(gi, L)
fr, (gx, gy) = DATA["test"]["frames"][c][t], Lb_test[c][t, :2]
S = 24
x0, y0 = int(np.clip(gx - S, 0, W - 2 * S)), int(np.clip(gy - S, 0, H - 2 * S))
fig, axes = plt.subplots(len(names), 3, figsize=(9.5, 1.85 * len(names)), gridspec_kw=dict(width_ratios=[1.8, 1.8, 1]))
for r, n in enumerate(names):
    hm = TEST_HM[n][c][t].astype(np.float32) / 255
    pr = TEST_PRED[n][c][t]
    axes[r, 0].imshow(fr); axes[r, 0].add_patch(plt.Rectangle((x0, y0), 2 * S, 2 * S, fill=False, ec=PAL[1], lw=1))
    axes[r, 1].imshow(hm, cmap="magma", vmin=0, vmax=1)
    axes[r, 2].imshow(fr[y0:y0 + 2 * S, x0:x0 + 2 * S], interpolation="nearest")
    axes[r, 2].imshow(hm[y0:y0 + 2 * S, x0:x0 + 2 * S], cmap="magma", alpha=0.45, vmin=0, vmax=1, interpolation="nearest")
    axes[r, 2].add_patch(Circle((gx - x0, gy - y0), 3.5, fill=False, ec=PAL[2], lw=1.5))
    if not np.isnan(pr[0]):
        axes[r, 2].scatter([pr[0] - x0], [pr[1] - y0], marker="x", s=60, color="#ffffff", lw=2)
        axes[r, 2].set_xlim(-0.5, 2 * S - 0.5); axes[r, 2].set_ylim(2 * S - 0.5, -0.5)
        err = math.hypot(pr[0] - gx, pr[1] - gy); txt = f"err {err:.2f}px"
    else:
        txt = "no detection"
    for a in axes[r]:
        no_axes(a)
    axes[r, 2].set_title(txt, fontsize=8.5, loc="left")
    axes[r, 0].text(2, 12, n, color="#ffffff", fontsize=8, fontweight="bold", bbox=dict(fc="#00000088", ec="none", pad=1.5))
axes[0, 1].set_title("predicted heatmap", fontsize=8.5, loc="left")
fig.suptitle(f"Test clip {c}, frame {t}: speed {Lb_test[c][t,3]:.1f} px/frame, radius {Lb_test[c][t,4]:.1f} px", x=0.01, ha="left", fontweight="bold")
plt.tight_layout()
savefig(fig, "fig13_heatmaps_per_model")

### 9.2 Inside MDD: learned motion polarity maps (V5 vs. M2)
The positive polarity $P^+$ fires where the ball *arrives* and $P^-$ where it *leaves*. Together they encode the direction of travel. The M2 direction channel $A^+-A^-$ makes that explicit: a red/blue dipole aligned with the velocity. The learned slope $k$ and threshold $m$ show what the network chose to count as motion.

In [ ]:
def mdd_params(model):
    k, m = model.mdd.k_m()
    return k.detach().cpu().flatten().numpy(), m.detach().cpu().flatten().numpy()

for n in ["V5 (faithful)", "V5 +M1+M2"]:
    k, m = mdd_params(MODELS[n])
    print(f"{n:16s} k = {np.round(k, 2)}   m = {np.round(m, 3)}" +
          ("   (order: pair1+, pair1−, pair2+, pair2−)" if len(k) > 1 else "   (shared)"))

s0 = int(np.clip(t - 1, 0, L - 3))                       # 3-frame window containing t (also valid at clip ends)
j = t - s0                                                 # index of frame t inside the window
pj = min(j, 1)                                             # frame-pair index: (t→t+1) if possible, else (t−1→t)
pair_txt = "t→t+1" if pj == j else "t−1→t"
x_win = torch.from_numpy((DATA["test"]["frames"][c][s0:s0 + 3].astype(np.float32) / 255).transpose(0, 3, 1, 2).reshape(1, 9, H, W)).to(DEVICE)
with torch.no_grad():
    A5 = MODELS["V5 (faithful)"].mdd(x_win)[0].cpu().numpy()
    A2 = MODELS["V5 +M1+M2"].mdd(x_win)[0].cpu().numpy()
crop = lambda a: a[y0:y0 + 2 * S, x0:x0 + 2 * S]
panels = [("frame t (crop)", crop(DATA["test"]["frames"][c][min(t, L - 1)]), None),
          (f"V5: $A^+$ ({pair_txt})", crop(A5[pj, 0]), "Blues"), (f"V5: $A^-$ ({pair_txt})", crop(A5[pj, 1]), "Blues"),
          (f"M2: $A^+$ ({pair_txt})", crop(A2[pj, 0]), "Blues"), (f"M2: $A^-$ ({pair_txt})", crop(A2[pj, 1]), "Blues"),
          ("M2: direction $A^+−A^-$", crop(A2[pj, 0] - A2[pj, 1]), "RdBu_r")]
fig, axes = plt.subplots(1, len(panels), figsize=(16, 3.0))
for ax, (ttl, img, cm) in zip(axes, panels):
    if cm is None:
        ax.imshow(img, interpolation="nearest")
    else:
        ax.imshow(img, cmap=cm, vmin=-1 if cm == "RdBu_r" else 0, vmax=1, interpolation="nearest")
    no_axes(ax); ax.set_title(ttl, fontsize=9)
plt.tight_layout()
savefig(fig, "fig14_mdd_polarity_maps")

### 9.3 Inside R-STR: draft → residual → refined heatmap
The Transformer head predicts a residual Δ that is added to the (MDD-fused) draft. A useful head **sharpens the true peak and suppresses competing blobs**. We compare the faithful V5 head (trained without the MDD fusion) with our consistent slim head (M3).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 5.6))
for r, n in enumerate(["V5 (faithful)", "V5 +M1+M2+M3"]):
    mdl = MODELS[n]; mdl.eval()
    with torch.no_grad():
        out, parts = mdl(x_win, return_parts=True)
    ims = [("draft σ(D)", torch.sigmoid(parts["draft"])[0, j]), ("draft + MDD fusion σ(D_MDD)", torch.sigmoid(parts["d_in"])[0, j]),
           ("residual Δ", parts["delta"][0, j]), ("final σ(D_MDD + Δ)", torch.sigmoid(out)[0, j])]
    for k_, (ttl, im) in enumerate(ims):
        im = crop(im.cpu().numpy())
        if "Δ" in ttl and "+" not in ttl:
            v = max(1e-3, np.abs(im).max()); axes[r, k_].imshow(im, cmap="RdBu_r", vmin=-v, vmax=v)
        else:
            axes[r, k_].imshow(im, cmap="magma", vmin=0, vmax=1)
        no_axes(axes[r, k_]); axes[r, k_].set_title(ttl, fontsize=8.5)
        axes[r, k_].add_patch(Circle((gx - x0, gy - y0), 3.5, fill=False, ec=PAL[2], lw=1.2))
    axes[r, 0].text(-6, S, n, rotation=90, ha="right", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
savefig(fig, "fig15_rstr_decomposition")

### 9.4 Tracking demo (GIF)
Final pipeline (V5-Lite+ with M5) on a full test clip. Green ring = ground truth, orange dot + trail = our track (frames upscaled 2× in the CPU profile). Saved to `figures/tracking_demo.gif`.

In [ ]:
import imageio.v2 as imageio
c = int(np.argmax([np.sum(l[:, 2] > 0) for l in Lb_test]))
UP = 2 if W < 400 else 1                     # upscale small frames; keep the GIF small at 512x288
frames_out = []
for t in range(L):
    im = cv2.resize(DATA["test"]["frames"][c][t], (W * UP, H * UP), interpolation=cv2.INTER_NEAREST)
    lab_t = Lb_test[c][t]
    if lab_t[2] > 0:
        cv2.circle(im, (int(lab_t[0] * UP), int(lab_t[1] * UP)), 9, (27, 175, 122), 1, cv2.LINE_AA)
    trail = [FINAL_PRED[c][k] for k in range(max(0, t - 8), t + 1) if not np.isnan(FINAL_PRED[c][k][0])]
    for a, b in zip(trail[:-1], trail[1:]):
        cv2.line(im, (int(a[0] * UP), int(a[1] * UP)), (int(b[0] * UP), int(b[1] * UP)), (235, 104, 52), 1, cv2.LINE_AA)
    if not np.isnan(FINAL_PRED[c][t][0]):
        cv2.circle(im, (int(FINAL_PRED[c][t][0] * UP), int(FINAL_PRED[c][t][1] * UP)), 3, (235, 104, 52), -1, cv2.LINE_AA)
    cv2.putText(im, f"frame {t:02d}  {'occluded' if lab_t[2]==2 else ''}", (6, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    frames_out.append(im)
imageio.mimsave(FIG / "tracking_demo.gif", frames_out, duration=1 / 12, loop=0)
fig, axes = plt.subplots(1, 4, figsize=(15, 2.3))
for ax, t in zip(axes, np.linspace(5, L - 5, 4).astype(int)):
    ax.imshow(frames_out[t]); no_axes(ax); ax.set_title(f"frame {t}", fontsize=8.5, loc="left")
plt.tight_layout()
savefig(fig, "fig16_tracking_strip")

### 9.5 Real broadcast footage: fine-tune on games 1–8, test on games 9–10
Synthetic data teaches the models *what to look for*. Real broadcasts add compression, crowds, logos, camera cuts and real ball appearance. Every variant is therefore:

1. **Evaluated zero-shot**, i.e. trained on synthetic data only, on the real test games 9–10 (sim-to-real gap).
2. **Fine-tuned** from its synthetic checkpoint on real games 1–8 (`CFG['ft_steps']` steps at `CFG['ft_lr']`, same loss, same augmentation and same ×0.1 step decay). The checkpoint is selected on the held-out last clip of each game. The synthetic starting point is itself a candidate, so fine-tuning can never end up worse than zero-shot on validation.
3. **Re-evaluated** on games 9–10, with and without M5.

All variants get exactly the same real-data budget, so the ablation (M1–M5) can be read on real footage as well. This answers whether the changes still help on real broadcasts, not only on our generator. Results are also broken down by TrackNet's own visibility classes: easy, hard (blurred/small) and occluded.

In [ ]:
FT_MODELS, FT_HIST = {}, {}
if REAL is not None:
    for name, spec in VARIANTS.items():
        FT_MODELS[name], FT_HIST[name] = train_variant(
            name, spec, train=REAL["train"], val=REAL["val"], steps=CFG["ft_steps"], lr=CFG["ft_lr"],
            eval_every=CFG["ft_eval_every"], init=MODELS[name], suffix="_realft")
        print(f"[{name}] fine-tuned in {FT_HIST[name]['time_s']/60:.1f} min\n")
else:
    print("Skipped: no real dataset found (see §4).")

In [ ]:
if REAL is not None:
    te = REAL["test"]
    rows, REAL_PRED = [], {}
    for name, spec in VARIANTS.items():
        for stage, mdl in (("zero-shot", MODELS[name]), ("fine-tuned", FT_MODELS[name])):
            hm = predict_heatmaps(mdl, te)
            pr, cf = decode_all(hm, spec["decode"], CFG["det_thr"])
            del hm                                                        # ~0.5 GB per model at 512x288
            rows.append(dict(model=name, stage=stage, **evaluate(pr, te["labels"])))
            if stage == "fine-tuned":
                REAL_PRED[name] = pr
                REAL_PRED[name + " + M5"] = [kalman_refine(p, c_) for p, c_ in zip(pr, cf)]
                rows.append(dict(model=name, stage="fine-tuned + M5", **evaluate(REAL_PRED[name + " + M5"], te["labels"])))
    REAL_RES = pd.DataFrame(rows).set_index(["model", "stage"])
    REAL_RES.to_csv(RES / f"real_tennis_{PROFILE}.csv")
    REAL_FT = REAL_RES.xs("fine-tuned", level="stage").join(COST)
    REAL_DELTAS = delta_table(REAL_FT)
    save_table(REAL_RES[["f1", "prec", "rec", "acc", "err_mean", "FP1", "FP2", "FN"]], "tab03_real_results",
               f"Real TrackNet tennis, test games 9–10 (τ = {CFG['tol']:.1f} px)", col_width=0.85, first_col_width=4.2)
    display(REAL_RES[["f1", "prec", "rec", "acc", "err_mean", "FP1", "FP2", "FN"]].unstack("stage")["f1"]
            .style.format("{:.3f}").background_gradient(cmap="Blues", axis=None).set_caption("Test F1 on real games 9–10"))
    display(REAL_RES[["f1", "prec", "rec", "acc", "err_mean", "FP1", "FP2", "FN"]].style.format(
        {"f1": "{:.3f}", "prec": "{:.3f}", "rec": "{:.3f}", "acc": "{:.3f}", "err_mean": "{:.2f}"}))
    display(REAL_DELTAS.style.format("{:+.3f}").set_caption("Effect of each change on real footage (fine-tuned models)"))

In [ ]:
if REAL is not None:
    names = list(VARIANTS)
    raw_te = np.concatenate(REAL["test"]["vis_raw"]); lab_te = np.concatenate(REAL["test"]["labels"])
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.0), gridspec_kw=dict(width_ratios=[3, 2]))
    xs = np.arange(len(names)); w = 0.27
    for k, (stage, col) in enumerate([("zero-shot", MUTED), ("fine-tuned", PAL[0]), ("fine-tuned + M5", PAL[1])]):
        vals = [REAL_RES.loc[(n, stage), "f1"] if (n, stage) in REAL_RES.index else np.nan for n in names]
        axes[0].bar(xs + (k - 1) * w, vals, width=w - 0.02, color=col, label=stage)
    axes[0].set_xticks(xs, [n.replace(" (", "\n(").replace("V5 +", "V5\n+") for n in names], fontsize=7.5)
    axes[0].set(title=f"Real test games 9–10: F1 @ τ={CFG['tol']:.1f}px", ylim=(0, 1.05)); axes[0].legend(fontsize=8, ncol=3, loc="upper left")
    # hit-rate by TrackNet's own visibility class
    P_all = {n: np.concatenate(REAL_PRED[n]) for n in ["V2 (baseline)", "V5 (faithful)", FINAL_NAME + " + M5"]}
    classes = [("easy (1)", raw_te == 1), ("hard (2)", raw_te == 2), ("occluded (3)", raw_te == 3)]
    for k, (n, P) in enumerate(P_all.items()):
        hits = [np.mean(np.hypot(P[m, 0] - lab_te[m, 0], P[m, 1] - lab_te[m, 1]) <= CFG["tol"]) if m.any() else np.nan
                for _, m in classes]
        axes[1].bar(np.arange(3) + (k - 1) * w, hits, width=w - 0.02, color=[MUTED, PAL[0], PAL[1]][k], label=n + " (fine-tuned)")
    axes[1].set_xticks(np.arange(3), [f"{c}\n(n={m.sum()})" for c, m in classes], fontsize=8)
    axes[1].set(title="Real test hit-rate by TrackNet visibility class", ylim=(0, 1.05)); axes[1].legend(fontsize=7.5, loc="upper right")
    plt.tight_layout()
    savefig(fig, "fig17_real_tennis")

    # example test frames: ground truth vs. faithful V5 vs. ours (both fine-tuned)
    te = REAL["test"]
    cands = [(c, t) for c in range(len(te["frames"])) for t in range(len(te["frames"][c])) if te["vis_raw"][c][t] in (1, 2)]
    picks = [cands[i] for i in np.linspace(0, len(cands) - 1, 4).astype(int)] if cands else []
    if picks:
        fig, axes = plt.subplots(1, len(picks), figsize=(4.2 * len(picks), 2.9))
        for ax, (c, t) in zip(np.atleast_1d(axes), picks):
            ax.imshow(te["frames"][c][t]); no_axes(ax)
            gx, gy = te["labels"][c][t, :2]
            ax.add_patch(Circle((gx, gy), 8 * SC / 1.8, fill=False, ec=PAL[2], lw=1.6))
            for n, mk, col in (("V5 (faithful)", "x", PAL[0]), (FINAL_NAME + " + M5", "+", PAL[1])):
                p = REAL_PRED[n][c][t]
                if not np.isnan(p[0]):
                    ax.scatter([p[0]], [p[1]], marker=mk, s=70, color=col, lw=2)
            ax.set_title(f"{te['clips'][c]} · frame {t} · vis {te['vis_raw'][c][t]}", fontsize=8.5, loc="left")
        fig.suptitle("Real test frames (games 9–10): green ring = ground truth, blue ✕ = V5 (faithful), orange + = V5-Lite+ with M5",
                     x=0.01, ha="left", fontweight="bold", fontsize=10)
        plt.tight_layout()
        savefig(fig, "fig18_real_examples")

## 10. Changes, trade-offs & improvements
The table below is **generated from the measured results above**. The narrative analysis follows it.

In [ ]:
def change_summary(res, deltas, final_m5):
    """One row per modification (from consecutive ablation rungs) + M5 + the total V5 -> ours."""
    v5_, fin = res.loc["V5 (faithful)"], res.loc[FINAL_NAME]
    steps = [("M1 Gaussian target + soft-argmax", "V5 → V5+M1", "V5 (faithful) → V5 +M1"),
             ("M2 per-polarity MDD + direction", "V5+M1 → +M2", "V5 +M1 → V5 +M1+M2"),
             ("M3 consistent, slim R-STR head", "+M2 → +M3", "V5 +M1+M2 → V5 +M1+M2+M3"),
             ("M4 depthwise-separable backbone", "+M3 → +M4", "V5 +M1+M2+M3 → V5-Lite+ (M1–M4)")]
    rows = [dict(change=ch, vs=vs, **deltas.loc[key].to_dict()) for ch, vs, key in steps]
    rows.append(dict(change="M5 Kalman gating + ballistic gap fill", vs="M1–M4 → +M5",
                     **{"ΔF1": final_m5.f1 - fin.f1, "ΔPrecision": final_m5.prec - fin.prec, "ΔRecall": final_m5.rec - fin.rec,
                        "Δerr (px)": final_m5.err_mean - fin.err_mean, "Δparams (M)": 0.0, "ΔGFLOPs": 0.0, "ΔFPS": np.nan}))
    rows.append(dict(change="TOTAL: V5 → V5-Lite+ with M5", vs="V5 → ours",
                     **{"ΔF1": final_m5.f1 - v5_.f1, "ΔPrecision": final_m5.prec - v5_.prec, "ΔRecall": final_m5.rec - v5_.rec,
                        "Δerr (px)": final_m5.err_mean - v5_.err_mean, "Δparams (M)": fin.params_M - v5_.params_M,
                        "ΔGFLOPs": fin.GFLOPs - v5_.GFLOPs, "ΔFPS": fin.FPS - v5_.FPS}))
    return pd.DataFrame(rows).set_index("change")


final_m5 = M5.loc[(FINAL_NAME, "+ M5 (gating + gap filling)")]
v5 = RESULTS.loc["V5 (faithful)"]
summary = change_summary(RESULTS, DELTAS, final_m5)
REAL_SUMMARY = None
if REAL is not None:
    REAL_SUMMARY = change_summary(REAL_FT, REAL_DELTAS, REAL_RES.loc[(FINAL_NAME, "fine-tuned + M5")])
    REAL_SUMMARY.to_csv(RES / f"real_changes_summary_{PROFILE}.csv")
summary.to_csv(RES / f"changes_summary_{PROFILE}.csv")
save_table(summary, "tab04_changes_synthetic", "Effect of each change: synthetic test set", fmt="{:+.3f}", first_col_width=3.6, col_width=1.0)
if REAL_SUMMARY is not None:
    save_table(REAL_SUMMARY, "tab04_changes_real", "Effect of each change: real test games 9–10 (fine-tuned)", fmt="{:+.3f}",
               first_col_width=3.6, col_width=1.0)
display(summary.style.format({c: "{:+.3f}" for c in summary.columns if c != "vs"}, na_rep="—")
        .set_caption("Synthetic test set"))
if REAL_SUMMARY is not None:
    display(REAL_SUMMARY.style.format({c: "{:+.3f}" for c in REAL_SUMMARY.columns if c != "vs"}, na_rep="—")
            .set_caption("Real footage: games 9–10, fine-tuned models"))
print(f"\nParams: V5 {v5.params_M:.3f}M → ours {RESULTS.loc[FINAL_NAME].params_M:.3f}M "
      f"({RESULTS.loc[FINAL_NAME].params_M / v5.params_M:.1%}); GFLOPs: {v5.GFLOPs:.2f} → {RESULTS.loc[FINAL_NAME].GFLOPs:.2f} "
      f"({RESULTS.loc[FINAL_NAME].GFLOPs / v5.GFLOPs:.1%})")
with open(RES / f"all_results_{PROFILE}.json", "w") as fh:
    json.dump(dict(profile=PROFILE, cfg=CFG, results=RESULTS.reset_index().to_dict(orient="records"),
                   m5=M5.reset_index().to_dict(orient="records")), fh, indent=1, default=float)
def to_md(df, fmt):
    try:
        return df.to_markdown(floatfmt=fmt)          # needs the optional 'tabulate' package
    except ImportError:
        return "```\n" + df.to_string() + "\n```"
(RES / f"changes_measured_{PROFILE}.md").write_text(
    f"# Measured effect of each change (profile `{PROFILE}`, τ = {CFG['tol']:.1f}px)\n\n" + to_md(summary, "+.3f") +
    "\n\n## Full ablation table\n\n" + to_md(RESULTS, ".3f") +
    ("" if REAL_SUMMARY is None else
     "\n\n# Real TrackNet tennis footage (fine-tune games 1–8, test games 9–10)\n\n" + to_md(REAL_SUMMARY, "+.3f") +
     "\n\n## Zero-shot vs. fine-tuned vs. fine-tuned + M5\n\n" +
     to_md(REAL_RES[["f1", "prec", "rec", "acc", "err_mean", "FP1", "FP2", "FN"]], ".3f")) + "\n", encoding="utf-8")

### 10.1 What each change was expected to do, and how to read the result
| Change | Hypothesis (why it should help) | Expected cost / risk | Where to verify |
|---|---|---|---|
| **M1** Gaussian + soft-argmax | The peak sits on the true centre, so localisation error drops and F1 at strict τ rises. Smaller targets also separate the ball from nearby players and rackets. | A Gaussian puts less total "positive mass" on the target than a disk. That can **slow early training** and lower recall on the faintest balls. | Fig. 9 (F1 vs τ, error CDF), `err_mean` |
| **M2** per-polarity MDD + direction | Arrival and departure motion get separate thresholds. The explicit dipole channel gives the network the direction of travel for free, which should help fast balls. | +6 learnable scalars, +2 input channels (negligible). May overfit to synthetic motion statistics. | Fig. 11 (speed bins), Fig. 14 |
| **M3** consistent slim R-STR | Removing the train/test input mismatch should cut FP2 (spurious peaks where MDD fires on moving players) and improve precision. 4× fewer tokens makes the head ~4× cheaper. | A coarser residual (16-px patches) refines less finely. | `prec`, FP2 in Fig. 7; GFLOPs, FPS |
| **M4** depthwise-separable convs | ~5× fewer parameters and ~4.5× fewer FLOPs, suitable for edge deployment. | Lower capacity. On CPU, depthwise convs are memory-bound, so **FLOPs fall far more than latency does**. | Fig. 8 (Pareto), `params_M`, `FPS` |
| **M5** Kalman gating + gap fill | Gating removes isolated false positives (precision up). Ballistic interpolation recovers occluded frames (recall on `occluded` up). | It can reject true detections right after a hit or bounce, and it interpolates wrongly if a hit happens inside a gap. | §8 table, Fig. 12, occluded hit-rate |

The cell below turns the measured numbers into a per-change verdict, so the conclusions always match the actual run.

In [ ]:
def verdict(dF1, eps=0.005):
    return "✅ improves" if dF1 > eps else ("❌ hurts" if dF1 < -eps else "➖ neutral on")

def verdict_lines(table):
    out = []
    for change, r in table.iterrows():
        eff = "" if np.isnan(r["ΔFPS"]) else f"; params {r['Δparams (M)']:+.3f} M, GFLOPs {r['ΔGFLOPs']:+.2f}, FPS {r['ΔFPS']:+.1f}"
        out.append(f"* **{change}** ({r['vs']}): {verdict(r['ΔF1'])} F1 ({r['ΔF1']:+.3f}). Precision {r['ΔPrecision']:+.3f}, "
                   f"recall {r['ΔRecall']:+.3f}, localisation error {r['Δerr (px)']:+.2f} px{eff}.")
    return out

lines = [f"**Run profile `{PROFILE}`**: {W}×{H}, width {CFG['width']}, {CFG['steps']} steps × batch {CFG['batch']}, τ = {CFG['tol']:.1f}px.\n"]
lines.append("#### Synthetic test set")
lines += verdict_lines(summary)
best = RESULTS["f1"].idxmax()
lines.append(f"\n**Best network without post-processing:** {best} (F1 {RESULTS.loc[best,'f1']:.3f}, mean error {RESULTS.loc[best,'err_mean']:.2f} px). "
             f"**Final pipeline (V5-Lite+ + M5):** F1 {final_m5.f1:.3f} vs. faithful V5 {v5.f1:.3f}. "
             f"Occluded-frame hit-rate {occl_before:.3f} → {occl_after:.3f} with M5.")
if REAL_SUMMARY is not None:
    zs, ft = REAL_RES.xs("zero-shot", level="stage")["f1"], REAL_RES.xs("fine-tuned", level="stage")["f1"]
    lines.append("\n#### Real footage (TrackNet tennis, test games 9–10, after fine-tuning on games 1–8)")
    lines += verdict_lines(REAL_SUMMARY)
    lines.append(f"\nSim-to-real gap: mean zero-shot F1 {zs.mean():.3f} → fine-tuned {ft.mean():.3f}. "
                 f"Best fine-tuned model: {ft.idxmax()} (F1 {ft.max():.3f}); with M5 our final pipeline reaches "
                 f"F1 {REAL_RES.loc[(FINAL_NAME, 'fine-tuned + M5'), 'f1']:.3f} vs. faithful V5 {ft['V5 (faithful)']:.3f}.")
display(Markdown("\n".join(lines)))

### 10.2 Comparison with existing approaches
Rows marked *measured* come from this run: identical data, budget and evaluation. *Literature* F1 values are what each paper reports **on its own dataset and split**. They are shown for context only and are **not comparable** with each other or with our numbers.

In [ ]:
def measured(name, post=None):
    syn = (M5.loc[(name, "+ M5 (gating + gap filling)")] if post else RESULTS.loc[name])
    real = np.nan
    if REAL is not None:
        real = REAL_RES.loc[(name, "fine-tuned + M5" if post else "fine-tuned"), "f1"]
    return dict(params_M=COST.loc[name, "params_M"], GFLOPs=COST.loc[name, "GFLOPs"], FPS=COST.loc[name, "FPS"],
                **{"F1 synthetic": syn.f1, "F1 real 9–10": real, "error px": syn.err_mean})

lit = {"TrackNet V1 (2019)": 0.985, "TrackNetV2 (2020)": 0.9677, "TrackNetV3 (2023)": 0.975, "TrackNetV4 (2024)": 0.9581, "TrackNetV5 (2025)": 0.9859}
lit_src = {"TrackNet V1 (2019)": "tennis, frame split", "TrackNetV2 (2020)": "badminton (V5 paper)",
           "TrackNetV3 (2023)": "badminton (V4 paper)", "TrackNetV4 (2024)": "badminton (V5 paper)",
           "TrackNetV5 (2025)": "badminton (V5 paper)"}
rows = [dict(approach=k, source="literature", **{"lit. F1": v, "lit. dataset": lit_src[k]}) for k, v in lit.items()]
rows += [dict(approach="TrackNetV2 (our re-implementation)", source="measured", **measured("V2 (baseline)")),
         dict(approach="TrackNetV5 (our faithful re-implementation)", source="measured", **measured("V5 (faithful)")),
         dict(approach="TrackNetV5-Lite+ M1–M4 (ours)", source="measured", **measured(FINAL_NAME)),
         dict(approach="TrackNetV5-Lite+ M1–M5 (ours, full)", source="measured", **measured(FINAL_NAME, post=True))]
COMPARISON = pd.DataFrame(rows).set_index("approach")[["source", "lit. F1", "lit. dataset", "params_M",
                                                       "GFLOPs", "FPS", "F1 synthetic", "F1 real 9–10", "error px"]]
COMPARISON.to_csv(RES / f"comparison_{PROFILE}.csv")
display(COMPARISON.style.format({c: "{:.3f}" for c in COMPARISON.columns if c not in ("source", "lit. dataset")}, na_rep="—"))
save_table(COMPARISON, "tab05_comparison", "Comparison with existing approaches (lit. = reported by each paper on its own dataset, not comparable; measured = this run)",
           first_col_width=3.6, col_width=1.25)

## 11. Conclusions, limitations & future work
**Conclusions.**
* The TrackNet line shows a clear pattern: every big gain came from giving the network a better **temporal signal**. That started with multi-frame input (V1), continued with motion attention (V4) and polarity-aware motion (V5), and is reflected in our M2 and M5. Appearance alone cannot separate a 3-px ball from a 3-px crowd speck.
* **The target matters as much as the architecture.** V5's 60-px binary disk is a sensible choice for *detection* (it is easy to learn), but it limits *localisation*. The F1-vs-τ curve (Fig. 9) is the right instrument to expose this, and a single loose-τ F1 hides it.
* **R-STR must see the same input in training and inference.** We read the paper's equations as training the head on Dropout(Draft) but running it on Draft$_{MDD}$. Making these consistent (M3) is free, and it lets the fusion weight actually be learned.
* **Efficiency is not FLOPs.** Depthwise-separable convolutions (M4) cut parameters by ≈ 5× and FLOPs by ≈ 4.5×, but they barely improve CPU latency, because these kernels are memory-bound. Deployment claims need wall-clock measurements on the target hardware.
* A **zero-parameter physics prior** (M5) addresses the authors' own stated limitation (false positives in clutter) and also adds the occlusion recovery that V5 lacks relative to V3.

**Limitations.**
* Training is mostly synthetic, plus a short fine-tuning on 8 real tennis matches. Real-footage results come from only 2 test matches (≈ 3.7k frames), and the paper reports on a different (badminton) dataset, so absolute numbers are **not** comparable with the paper's. The *relative* effects of the changes are the meaningful result.
* The models are trained at reduced width and with a short schedule, for compute reasons, and every result is a **single seed**. Differences below ~0.01 F1 should be treated as noise.
* The R-STR head's exact size (layers, heads, dim) is not specified in the paper, so the "faithful" head is our best reading of it.

**Future work.** Train from scratch on real data and add the TrackNetV2 shuttlecock dataset; cross-validate over all 10 tennis matches; run multiple seeds with confidence intervals; learn the M5 dynamics (e.g. a small GRU smoother as in TrackNetV3's rectification); try mixed precision and TensorRT for real-time edge deployment; and use a longer temporal window, which V5's authors suggest as the way to reduce false positives further.

### References
1. Y.-C. Huang, I-N. Liao, C.-H. Chen, T.-U. İk, W.-C. Peng. *TrackNet: A Deep Learning Network for Tracking High-speed and Tiny Objects in Sports Applications.* IEEE AVSS 2019. arXiv:1907.03698
2. N.-E. Sun et al. *TrackNetV2: Efficient Shuttlecock Tracking Network.* ICPAI 2020.
3. Y.-J. Chen, Y.-S. Wang. *TrackNetV3: Enhancing ShuttleCock Tracking with Augmentations and Trajectory Rectification.* ACM MM Asia 2023.
4. A. Raj, L. Wang, T. Gedeon. *TrackNetV4: Enhancing Fast Sports Object Tracking with Motion Attention Maps.* arXiv:2409.14543, 2024 (ICASSP 2025).
5. H. Tang, Y. Chen, L. Jiang. *TrackNetV5: Residual-Driven Spatio-Temporal Refinement and Motion Direction Decoupling for Fast Object Tracking.* arXiv:2512.02789, 2025.
6. O. Ronneberger, P. Fischer, T. Brox. *U-Net: Convolutional Networks for Biomedical Image Segmentation.* MICCAI 2015.
7. G. Bertasius, H. Wang, L. Torresani. *Is Space-Time Attention All You Need for Video Understanding?* (TimeSformer). ICML 2021.
8. T.-Y. Lin et al. *Focal Loss for Dense Object Detection.* ICCV 2017; X. Zhou et al. *Objects as Points* (CenterNet). 2019.
9. A. G. Howard et al. *MobileNets: Efficient CNNs for Mobile Vision Applications.* 2017 (depthwise-separable convolutions).
10. R. E. Kalman. *A New Approach to Linear Filtering and Prediction Problems.* 1960; H. Rauch, F. Tung, C. Striebel. *Maximum likelihood estimates of linear dynamic systems.* AIAA J. 1965.

## 12. Report assets
The cells below record the tools and environment used, then check that every asset the project report needs was produced:
- `results/REPORT_ASSETS.md` maps each report section to its files and key numbers.
- `results/report_assets_check_<profile>.json` lists anything missing.

In [ ]:
import platform, sys
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)"
syn_min = sum(h["time_s"] for h in HIST.values()) / 60
ft_min = sum(h["time_s"] for h in FT_HIST.values()) / 60 if REAL is not None else 0.0
ENV = pd.DataFrame([
    ("Platform", "Kaggle Notebook" if IS_KAGGLE else platform.platform()),
    ("Hardware", f"{gpu} | CPU threads {os.cpu_count()}"),
    ("Python", sys.version.split()[0]),
    ("PyTorch", f"{torch.__version__} (CUDA {torch.version.cuda or '—'}, cuDNN {torch.backends.cudnn.version() or '—'})"),
    ("NumPy / pandas", f"{np.__version__} / {pd.__version__}"),
    ("OpenCV / Matplotlib", f"{cv2.__version__} / {mpl.__version__}"),
    ("FLOP counting", "torch.utils.flop_counter.FlopCounterMode"),
    ("Profile", f"{PROFILE} — {W}×{H}, width {CFG['width']}, {CFG['steps']} steps (+{CFG['ft_steps']} fine-tune) × batch {CFG['batch']}"),
    ("Synthetic training (6 models)", f"{syn_min:.1f} min"),
    ("Real fine-tuning (6 models)", f"{ft_min:.1f} min" if REAL is not None else "skipped (no real data)"),
    ("Whole notebook", f"{(time.time() - T_START) / 60:.1f} min"),
], columns=["item", "value"]).set_index("item")
display(ENV)
(RES / "environment.md").write_text("# Tools and environment\n\n" + ENV.to_markdown() +
                                     "\n\nLibraries: PyTorch, NumPy, pandas, OpenCV, Matplotlib, imageio, Jupyter; "
                                     "run on Kaggle Notebooks (GPU) and tested locally with pytest.\n", encoding="utf-8")
save_table(ENV, "tab06_environment", "Tools and environment", fmt="{}", col_width=6.5, first_col_width=2.6)

In [ ]:
def F_(name):
    return FIG / name

REPORT = {
    "1. Problem statement": [("notebook §1.1 (text)", None)],
    "2. Gaps identified": [("notebook §1.4 (table) + CHANGES.md in the repo", None)],
    "3. Proposed architecture diagram": [("block diagram", F_("fig02_pipeline_v5_vs_ours.png")),
                                         ("detailed diagram", F_("fig19_architecture_detailed.png")),
                                         ("layer table", RES / f"model_summary_{PROFILE}.md")],
    "4. Algorithmic steps": [("notebook §5.2 (text)", None)],
    "5. Dataset used": [("synthetic samples", F_("fig03_dataset_samples.png")), ("synthetic statistics", F_("fig04_dataset_stats.png")),
                        ("training targets", F_("fig05_targets_disk_vs_gaussian.png")), ("dataset table", F_("tab00_datasets.png")),
                        ("dataset summary", RES / f"dataset_summary_{PROFILE}.md")] +
                       ([("real samples", F_("fig20_real_dataset.png"))] if REAL is not None else []),
    "6. Tools used": [("environment table", F_("tab06_environment.png")), ("environment", RES / "environment.md")],
    "7. Screenshots of results": [("synthetic ablation table", F_("tab02_synthetic_ablation.png")),
                                  ("training curves", F_("fig06_training_curves.png")), ("ablation bars", F_("fig07_ablation_bars.png")),
                                  ("heatmaps per model", F_("fig13_heatmaps_per_model.png")), ("tracking strip", F_("fig16_tracking_strip.png")),
                                  ("tracking GIF", F_("tracking_demo.gif")), ("M5 before/after", F_("fig12_kalman_m5.png"))] +
                                 ([("real results table", F_("tab03_real_results.png")), ("real results chart", F_("fig17_real_tennis.png")),
                                   ("real example frames", F_("fig18_real_examples.png"))] if REAL is not None else []),
    "8. Analysis & difference from existing approaches": [("comparison table", F_("tab05_comparison.png")),
                                                         ("per-change effect (synthetic)", F_("tab04_changes_synthetic.png")),
                                                         ("accuracy vs cost", F_("fig08_accuracy_vs_cost.png")),
                                                         ("tolerance / error CDF", F_("fig09_tolerance_and_error_cdf.png")),
                                                         ("robustness", F_("fig11_robustness.png")), ("MDD maps", F_("fig14_mdd_polarity_maps.png")),
                                                         ("R-STR decomposition", F_("fig15_rstr_decomposition.png")),
                                                         ("all numbers", RES / f"changes_measured_{PROFILE}.md")] +
                                                        ([("per-change effect (real)", F_("tab04_changes_real.png"))] if REAL is not None else []),
    "9. Code link": [("GitHub repo / public Kaggle notebook URL (added by the student)", None)],
    "10. Novel contribution": [("notebook §1.4.2 (text)", None), ("model cost table", F_("tab01_model_cost.png"))],
    "11. Individual contribution": [("written by the student", None)],
    "12. Learning outcome": [("notebook §11 conclusions (basis)", None)],
}
missing, lines = [], [f"# Report assets (profile `{PROFILE}`, generated {time.strftime('%Y-%m-%d %H:%M')})\n",
                      "| Report section | Asset | File | Status |", "|---|---|---|---|"]
for section, items in REPORT.items():
    for label, path in items:
        if path is None:
            status, shown = "text", "—"
        else:
            ok = Path(path).exists(); status = "✅" if ok else "❌ missing"; shown = str(Path(path).relative_to(ROOT)) if ok else str(path)
            if not ok:
                missing.append(f"{section}: {label} ({path})")
        lines.append(f"| {section} | {label} | `{shown}` | {status} |")
best_syn = M5.loc[(FINAL_NAME, "+ M5 (gating + gap filling)")]
lines += ["", "## Key numbers", f"- Synthetic test F1 @ τ={CFG['tol']:.1f}px: faithful V5 {RESULTS.loc['V5 (faithful)', 'f1']:.3f} → "
          f"ours (M1–M5) {best_syn.f1:.3f}",
          f"- Params: V5 {COST.loc['V5 (faithful)', 'params_M']:.3f} M → ours {COST.loc[FINAL_NAME, 'params_M']:.3f} M; "
          f"GFLOPs {COST.loc['V5 (faithful)', 'GFLOPs']:.2f} → {COST.loc[FINAL_NAME, 'GFLOPs']:.2f}"]
if REAL is not None:
    lines.append(f"- Real games 9–10 F1: faithful V5 (fine-tuned) {REAL_RES.loc[('V5 (faithful)', 'fine-tuned'), 'f1']:.3f} → "
                 f"ours + M5 {REAL_RES.loc[(FINAL_NAME, 'fine-tuned + M5'), 'f1']:.3f}")
(RES / "REPORT_ASSETS.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
(RES / f"report_assets_check_{PROFILE}.json").write_text(json.dumps(dict(missing=missing, real_data=REAL is not None), indent=1))
display(Markdown("\n".join(lines)))
print("All report assets present." if not missing else "MISSING:\n" + "\n".join(missing))

### Export (Kaggle)
Bundles the figures, result tables and trained checkpoints into `tracknet_outputs.zip`, so they can be downloaded from Kaggle's *Output* tab and dropped back into the repository.

In [ ]:
import zipfile
if IS_KAGGLE or os.environ.get("TN_ZIP") == "1":
    with zipfile.ZipFile(ROOT / "tracknet_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for doc in ("README.md", "CHANGES.md"):
            if (ROOT / doc).exists():
                zf.write(ROOT / doc, doc)
        for d in (FIG, RES, CKPT):
            for f in sorted(d.rglob("*")):
                if f.is_file():
                    zf.write(f, f.relative_to(ROOT))
    print(f"wrote {ROOT / 'tracknet_outputs.zip'} ({(ROOT / 'tracknet_outputs.zip').stat().st_size / 1e6:.1f} MB)")
else:
    print("Not on Kaggle: outputs are already in figures/, results/ and checkpoints/.")